# Composed Plots
> Combine transcript structure plots with quantitative visualizations

In [ ]:
#| default_exp composed_plots

In [ ]:
#| export
from __future__ import annotations

from typing import Any,  List, Optional, Tuple, Dict, Literal
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.sparse import issparse

# Import from standalone modules
from allos.transcript_plots import TranscriptPlots, PanelLayout, make_transcript_panel_layout, add_heatmap_to_layout, add_dotpanel_to_layout, add_categorical_band_to_layout, add_quant_band_to_layout, _PALETTE_WONG, _PALETTE_TOL_MUTED, _DEFAULT_COLORS
from allos.color_palette import ghibli as _PALETTE_GHIBLI
from allos.quant_plots import _compute_group_matrix_from_adata, _compute_dot_matrices_from_adata, _compute_replicate_psi_from_adata, _standalone_replicate_plot, _dense_X
from allos.coord_plots import calculate_density, get_coords_from_adata, get_expression_from_adata, _versionless

In [ ]:
#| export
# =====================================================================
# Panel alignment helpers
# =====================================================================

def _compute_panel_bounds_for_alignment(row_pitch: float, n_rows: int) -> Tuple[float, float]:
    """
    Compute the correct y-bounds for the panel so that dots at integer Y coords
    align with transcript intron lines.
    
    Transcripts are drawn at y = 0, -row_pitch, -2*row_pitch, ...
    We want panel y=0 to align with first transcript (y=0),
    panel y=1 to align with second transcript (y=-row_pitch), etc.
    
    Returns (y_top, y_bottom) in data coordinates for bbox_to_anchor.
    """
    if n_rows <= 0:
        return 0.3, -0.2
    y_top = 0.5 * row_pitch
    y_bottom = -row_pitch * (n_rows - 0.5)
    return y_top, y_bottom


def make_aligned_transcript_panel_layout(
    tp: TranscriptPlots,
    transcripts_ids: List[str],
    *,
    n_panel_cols: int,
    fig_width: float = None,
    draw_cds: bool = True,
    show_ruler: bool = True,
    bands_frac_top: float = 0.0,
    bands_frac_bottom: float = 0.0,
    fig_height: float | None = None,
    panel_width: float | None = None,
) -> PanelLayout:
    """
    Create a transcript panel layout with PERFECT alignment.
    
    This wrapper around make_transcript_panel_layout fixes:
    1. Panel bounds so dots/cells align exactly with transcript intron lines
    2. Band positions (top/bottom) to attach to the realigned panel
    3. Legend panel height to match the actual content height
    """
    # Get the standard layout
    layout = make_transcript_panel_layout(
        tp,
        transcripts_ids,
        n_panel_cols=n_panel_cols,
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ruler,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=bands_frac_bottom,
        fig_height=fig_height,
        panel_width=panel_width,
    )
    
    n_rows = len(layout.tids)
    if n_rows <= 0:
        return layout
    
    # Calculate correct alignment bounds
    y_top, y_bottom = _compute_panel_bounds_for_alignment(tp.row_pitch, n_rows)
    
    # Get panel dimensions
    panel_w = tp._panel_width_for(n_panel_cols)
    x0 = tp.panel_x0
    
    # ---- Fix panel alignment ----
    layout.ax_panel.remove()
    new_panel = inset_axes(
        layout.ax_main,
        width="100%",
        height="100%",
        loc="lower left",
        bbox_to_anchor=(x0, y_bottom, panel_w, y_top - y_bottom),
        bbox_transform=layout.ax_main.transData,
        borderpad=0,
    )
    new_panel.set_xlim(-0.5, n_panel_cols - 0.5)
    new_panel.set_ylim(-0.5, n_rows - 0.5)
    new_panel.invert_yaxis()
    for s in new_panel.spines.values():
        s.set_visible(False)
    new_panel.tick_params(axis="both", length=0)
    layout.ax_panel = new_panel
    
    # ---- Fix band positions to attach to realigned panel ----
    # Calculate band height in data coords (proportional to panel height)
    panel_height = y_top - y_bottom
    
    # Top band should sit directly above the panel
    if layout.ax_band_top is not None:
        band_data_height = bands_frac_top * panel_height / (1.0 - bands_frac_top - bands_frac_bottom) if bands_frac_top > 0 else 0.1
        
        layout.ax_band_top.remove()
        new_band_top = inset_axes(
            layout.ax_main,
            width="100%",
            height="100%",
            loc="lower left",
            bbox_to_anchor=(x0, y_top, panel_w, band_data_height),
            bbox_transform=layout.ax_main.transData,
            borderpad=0,
        )
        new_band_top.set_xlim(-0.5, n_panel_cols - 0.5)
        new_band_top.set_ylim(0.0, 1.0)
        new_band_top.set_facecolor('none')
        for s in new_band_top.spines.values():
            s.set_visible(False)
        new_band_top.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
        layout.ax_band_top = new_band_top
    
    # Bottom band should sit directly below the panel
    if layout.ax_band_bottom is not None:
        band_data_height = bands_frac_bottom * panel_height / (1.0 - bands_frac_top - bands_frac_bottom) if bands_frac_bottom > 0 else 0.1
        
        layout.ax_band_bottom.remove()
        new_band_bottom = inset_axes(
            layout.ax_main,
            width="100%",
            height="100%",
            loc="lower left",
            bbox_to_anchor=(x0, y_bottom - band_data_height, panel_w, band_data_height),
            bbox_transform=layout.ax_main.transData,
            borderpad=0,
        )
        new_band_bottom.set_xlim(-0.5, n_panel_cols - 0.5)
        new_band_bottom.set_ylim(0.0, 1.0)
        new_band_bottom.set_facecolor('none')
        for s in new_band_bottom.spines.values():
            s.set_visible(False)
        new_band_bottom.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
        layout.ax_band_bottom = new_band_bottom
    
    # ---- Fix legend panel height ----
    if layout.ax_legend is not None:
        fig = layout.fig
        ax_main = layout.ax_main
        main_pos = ax_main.get_position()
        fig.canvas.draw_idle()
        ylim = ax_main.get_ylim()
        y_range = ylim[1] - ylim[0]
        frac_top = (y_top - ylim[0]) / y_range
        frac_bottom = (y_bottom - ylim[0]) / y_range
        y_top_fig = main_pos.y0 + frac_top * main_pos.height
        y_bottom_fig = main_pos.y0 + frac_bottom * main_pos.height
        padding = 0.02
        y_top_fig = min(y_top_fig + padding, 0.95)
        y_bottom_fig = max(y_bottom_fig - padding, 0.05)
        legend_pos = layout.ax_legend.get_position()
        new_height = y_top_fig - y_bottom_fig
        layout.ax_legend.set_position([legend_pos.x0, y_bottom_fig, legend_pos.width, new_height])
        
        if layout.ax_cbar is not None:
            cbar_pos = layout.ax_cbar.get_position()
            cbar_height = min(cbar_pos.height, new_height * 0.6)
            cbar_y = y_bottom_fig + (new_height - cbar_height) / 2
            layout.ax_cbar.set_position([cbar_pos.x0, cbar_y, cbar_pos.width, cbar_height])
    
    # Store raw data-coordinate parameters for band drawing.
    # Figure coordinates are computed later (after heatmap is added) to avoid
    # computing positions before the layout has settled.
    _bth = bands_frac_top * panel_height / (1.0 - bands_frac_top - bands_frac_bottom) if bands_frac_top > 0 and layout.ax_band_top is not None else 0.0
    _bbh = bands_frac_bottom * panel_height / (1.0 - bands_frac_top - bands_frac_bottom) if bands_frac_bottom > 0 and layout.ax_band_bottom is not None else 0.0
    layout._band_data_params = {
        'x0': x0, 'panel_w': panel_w, 'n_cols': n_panel_cols,
        'y_top': y_top, 'y_bottom': y_bottom,
        'bth': _bth, 'bbh': _bbh,
        'top_frac': _bth / panel_height if panel_height > 0 else 0.0,
        'bot_frac': _bbh / panel_height if panel_height > 0 else 0.0,
    }
    return layout


# =====================================================================
# Legend Panel Builder - unified legend layout system
# =====================================================================

@dataclass
class CategoricalSlot:
    """Slot for categorical legend (group colors)."""
    groups: List[str]
    color_map: Dict[str, str]
    title: str = "Groups"


@dataclass
class ColorbarSlot:
    """Slot for continuous colorbar."""
    norm: mpl.colors.Normalize
    cmap: mpl.colors.Colormap
    label: str = ""


@dataclass
class DotSizeSlot:
    """Slot for dot size legend."""
    sizes: List[float] = None
    labels: List[str] = None
    title: str = "Prevalence\n(dot size)"


class LegendPanelBuilder:
    """
    Builder for creating consistent legend panels across different composed plots.
    
    Automatically stacks legend elements vertically with proper spacing:
    - Categorical legends (group colors)
    - Colorbars (PSI, GEX, etc.)
    - Dot size legends (prevalence)
    """
    
    def __init__(self, ax: Optional[plt.Axes]):
        self.ax = ax
        self._slots: List[Any] = []
    
    def add_categorical(self, groups: List[str], color_map: Dict[str, str], title: str = "Groups"):
        """Add a categorical legend showing group colors."""
        self._slots.append(CategoricalSlot(groups=groups, color_map=color_map, title=title))
        return self
    
    def add_colorbar(self, norm: mpl.colors.Normalize, cmap, label: str = ""):
        """Add a continuous colorbar."""
        if isinstance(cmap, str):
            cmap = mpl.cm.get_cmap(cmap)
        self._slots.append(ColorbarSlot(norm=norm, cmap=cmap, label=label))
        return self
    
    def add_dot_size_legend(self, sizes: List[float] = None, labels: List[str] = None, 
                           title: str = "Prevalence\n(dot size)"):
        """Add a dot size legend."""
        self._slots.append(DotSizeSlot(sizes=sizes, labels=labels, title=title))
        return self
    
    def render(self, spacing: float = 0.04, pad: float = 0.02):
        """Render all slots vertically stacked.  DotSizeSlot gets 3x height."""
        if self.ax is None or not self._slots:
            return
        
        self.ax.clear()
        self.ax.set_axis_off()
        
        n_slots = len(self._slots)
        weights = [1.5 if isinstance(s, DotSizeSlot) else 1.0 for s in self._slots]
        total_w = sum(weights)
        avail = 1.0 - (n_slots + 1) * spacing
        unit_h = avail / total_w
        current_y = 1.0 - spacing
        
        for slot, w in zip(self._slots, weights):
            slot_height = unit_h * w
            if isinstance(slot, CategoricalSlot):
                self._render_categorical(slot, current_y, slot_height, pad)
            elif isinstance(slot, ColorbarSlot):
                self._render_colorbar(slot, current_y, slot_height, pad)
            elif isinstance(slot, DotSizeSlot):
                self._render_dot_size(slot, current_y, slot_height, pad)
            current_y -= slot_height + spacing
    
    def _render_categorical(self, slot: CategoricalSlot, y_top: float, height: float, pad: float):
        from matplotlib.lines import Line2D
        cat_ax = inset_axes(
            self.ax, width="100%", height="100%",
            loc="upper left",
            bbox_to_anchor=(pad, y_top - height, 1.0 - pad, height),
            bbox_transform=self.ax.transAxes,
            borderpad=0,
        )
        cat_ax.set_axis_off()
        handles = [
            Line2D([0], [0], marker="s", linestyle="",
                   markersize=6, markerfacecolor=slot.color_map[g],
                   markeredgecolor="black", label=str(g))
            for g in slot.groups
        ]
        cat_ax.legend(handles, [str(g) for g in slot.groups],
                      loc="upper left", frameon=False, fontsize=7,
                      title=slot.title, title_fontsize=8)
    
    def _render_colorbar(self, slot: ColorbarSlot, y_top: float, height: float, pad: float):
        cb_ax = inset_axes(
            self.ax, width="45%", height="100%",
            loc="upper left",
            bbox_to_anchor=(pad, y_top - height, 0.45, height),
            bbox_transform=self.ax.transAxes,
            borderpad=0,
        )
        cb = mpl.colorbar.ColorbarBase(cb_ax, cmap=slot.cmap, norm=slot.norm, orientation="vertical")
        cb.set_label(slot.label, fontsize=11, fontweight="bold")
        cb.ax.tick_params(labelsize=10)
    
    def _render_dot_size(self, slot: DotSizeSlot, y_top: float, height: float, pad: float):
        sizes = slot.sizes or [0.25, 0.5, 0.75, 1.0]
        labels = slot.labels or ["25%", "50%", "75%", "100%"]

        dot_ax = inset_axes(
            self.ax, width="100%", height="100%",
            loc="upper left",
            bbox_to_anchor=(pad, y_top - height, 1.0 - pad, height),
            bbox_transform=self.ax.transAxes,
            borderpad=0,
        )
        dot_ax.set_axis_off()
        dot_ax.set_xlim(0, 1)
        dot_ax.set_ylim(0, 1)

        # Sizes matching add_dotpanel_to_layout: S = 16 + frac * 164
        sizes_desc  = list(reversed(sizes))
        labels_desc = list(reversed(labels))
        s_vals = [16.0 + sz * 164.0 for sz in sizes_desc]

        n = len(sizes_desc)
        title_frac = 0.25          # top 25% for 2-line title (~22 pts at fontsize 9)
        dot_start  = 1.0 - title_frac
        step       = dot_start / n  # equal cell per dot

        # Compute slot height in typographic pts  -  no canvas.draw() needed.
        # self.ax.get_position() is set at axis creation and is always valid.
        fig           = self.ax.get_figure()
        ax_h_typopt   = self.ax.get_position().height * fig.get_size_inches()[1] * 72.0
        slot_h_typopt = height * ax_h_typopt           # height = slot fraction of legend axis
        step_typopt   = dot_start * slot_h_typopt / n  # physical pts per dot cell

        # Scale dots so the largest fits its cell with a 5pt gap on each side.
        r_max_true = np.sqrt(s_vals[0] / np.pi)
        r_max_fit  = max(1.0, (step_typopt - 5.0) / 2.0)
        scale      = min(1.0, r_max_fit / r_max_true)
        s_vals     = [s * scale ** 2 for s in s_vals]

        title_text = slot.title.replace("\\n", "\n")
        dot_ax.text(0.5, 1.0, title_text,
                    transform=dot_ax.transAxes, fontsize=9,
                    ha="center", va="top", fontweight="bold")

        for i, (s_pts, lbl) in enumerate(zip(s_vals, labels_desc)):
            y_c = dot_start - (i + 0.5) * step
            dot_ax.scatter([0.22], [y_c], s=s_pts, c="gray", edgecolors="black",
                           linewidths=0.5, transform=dot_ax.transAxes, clip_on=False,
                           zorder=3)
            dot_ax.text(0.42, y_c, lbl, fontsize=10, fontweight="bold", ha="left", va="center",
                        transform=dot_ax.transAxes, zorder=4)

    def hide(self):
        """Hide the legend panel."""
        if self.ax is not None:
            self.ax.set_visible(False)


def make_group_color_map(groups: List[str], cmap_name: str = "tab20") -> Dict[str, str]:
    """Create a color map for categorical groups."""
    cmap = mpl.cm.get_cmap(cmap_name)
    return {g: mpl.colors.to_hex(cmap(i / max(1, len(groups) - 1))) for i, g in enumerate(groups)}


# =====================================================================
# Shared helpers  -  isoform selection, bands, tiled subaxes
# =====================================================================

def _select_isoforms(
    adata, gene_id, group_col, *,
    top_n=2, estimator="pseudobulk",
    dirichlet_alpha=0.5, epsilon=1e-6,
    visible_groups=None,
):
    """Select top isoforms and filter by visible groups.

    Returns (iso_ids, groups, V) or None if empty.
    """
    iso_ids, groups, V = _compute_group_matrix_from_adata(
        adata, gene_id, group_col,
        top_n=top_n, estimator=estimator,
        dirichlet_alpha=dirichlet_alpha, epsilon=epsilon,
    )
    if not iso_ids or V.size == 0:
        return None
    if visible_groups is not None:
        keep = [groups.index(g) for g in visible_groups if g in groups]
        if not keep:
            return None
        V = V[:, keep]
        groups = [groups[i] for i in keep]
    return iso_ids, groups, V


def _select_isoforms_dot(
    adata, gene_id, group_col, *,
    top_n=2, estimator="pseudobulk",
    dirichlet_alpha=0.5, epsilon=1e-6,
    visible_groups=None,
    size_mode="prevalence", psi_threshold=0.05,
):
    """Select top isoforms for dot plots (returns V and S matrices).

    Returns (iso_ids, groups, V, S) or None if empty.
    """
    iso_ids, groups, V, S = _compute_dot_matrices_from_adata(
        adata, gene_id, group_col,
        top_n=top_n, estimator=estimator,
        dirichlet_alpha=dirichlet_alpha, epsilon=epsilon,
        size_mode=size_mode, psi_threshold=psi_threshold,
    )
    if not iso_ids or V.size == 0:
        return None
    if visible_groups is not None:
        keep = [groups.index(g) for g in visible_groups if g in groups]
        if not keep:
            return None
        V = V[:, keep]
        S = S[:, keep]
        groups = [groups[k] for k in keep]
    return iso_ids, groups, V, S


def _compute_band_fracs(
    add_gex_band, add_group_color_band,
    gex_band_position="top", group_band_position="bottom",
    bands_frac=0.03,
):
    """Compute bands_frac_top and bands_frac_bottom from band flags."""
    gpos = (gex_band_position or "top").lower()
    cpos = (group_band_position or "bottom").lower()
    bands_frac_top = bands_frac if (
        (add_gex_band and gpos == "top") or
        (add_group_color_band and cpos == "top")
    ) else 0.0
    bands_frac_bottom = bands_frac if (
        (add_gex_band and gpos == "bottom") or
        (add_group_color_band and cpos == "bottom")
    ) else 0.0
    return bands_frac_top, bands_frac_bottom, gpos, cpos


def _compute_gex_vals(adata, gene_id, groups, group_col, epsilon=1e-6):
    """Compute CP10k gene expression values per group."""
    X = _dense_X(adata.X)
    if "geneId" in adata.var:
        m = adata.var["geneId"].astype(str).values == str(gene_id)
    else:
        m = np.zeros(adata.n_vars, bool)

    if not np.any(m):
        return np.zeros(len(groups), float)

    Xg = X[:, m]
    if Xg.ndim == 2 and Xg.shape[1] > 1:
        Xg = Xg.sum(1)
    lib_tot = X.sum(1) + epsilon

    all_groups = adata.obs[group_col].astype(str).values
    vals = []
    for g in groups:
        sel = (all_groups == g)
        if not sel.any():
            vals.append(0.0)
        else:
            cp10k = (Xg[sel] / lib_tot[sel]) * 1e4
            vals.append(float(np.nanmean(cp10k)))
    return np.asarray(vals, float)


def _populate_bands(
    layout, adata, gene_id, groups, group_col, *,
    add_gex_band=False, add_group_color_band=False,
    group_color_map=None,
    gex_band_position="top", group_band_position="bottom",
    epsilon=1e-6, tick_pad=35, font_scale=1.0,
):
    """Add categorical and/or GEX bands to layout.

    Returns (cat_used, quant_used, group_color_map, quant_norm, quant_cmap).
    """
    _fs = lambda x: int(round(x * font_scale))
    gpos = (gex_band_position or "top").lower()
    cpos = (group_band_position or "bottom").lower()

    cat_used = False
    quant_used = False
    quant_norm = None
    quant_cmap = None

    if not (add_gex_band or add_group_color_band) or len(groups) == 0:
        return cat_used, quant_used, group_color_map, quant_norm, quant_cmap

    top_slots = int(add_gex_band and gpos == "top") + \
                int(add_group_color_band and cpos == "top")
    bottom_slots = int(add_gex_band and gpos == "bottom") + \
                   int(add_group_color_band and cpos == "bottom")

    slot_top = 0
    slot_bottom = 0

    if add_group_color_band:
        if group_color_map is None:
            cmap_cat = mpl.cm.get_cmap("tab20")
            group_color_map = {
                g: mpl.colors.to_hex(cmap_cat(i / max(1, len(groups))))
                for i, g in enumerate(groups)
            }

        if cpos == "top":
            add_categorical_band_to_layout(
                layout, groups, color_map=group_color_map,
                slot=slot_top, n_slots=max(1, top_slots), which="top",
            )
            slot_top += 1
        else:
            add_categorical_band_to_layout(
                layout, groups, color_map=group_color_map,
                slot=slot_bottom, n_slots=max(1, bottom_slots), which="bottom",
            )
            slot_bottom += 1
        cat_used = True

    if add_gex_band:
        vals = _compute_gex_vals(adata, gene_id, groups, group_col, epsilon=epsilon)
        if gpos == "top":
            quant_norm, quant_cmap = add_quant_band_to_layout(
                layout, vals, cmap="viridis",
                slot=slot_top, n_slots=max(1, top_slots), which="top",
            )
            slot_top += 1
        else:
            quant_norm, quant_cmap = add_quant_band_to_layout(
                layout, vals, cmap="viridis",
                slot=slot_bottom, n_slots=max(1, bottom_slots), which="bottom",
            )
            slot_bottom += 1
        quant_used = True

    if bottom_slots > 0:
        layout.ax_panel.tick_params(axis="x", pad=tick_pad)

    return cat_used, quant_used, group_color_map, quant_norm, quant_cmap


def _make_tiled_subaxes(ax_panel, n_items, max_cols=2, pad_x=0.05, pad_y=0.05):
    """Create a grid of inset axes inside ax_panel for tiled plots.

    Returns list of Axes in row-major order.
    """
    ax_panel.set_axis_off()
    ncols = min(max_cols, n_items)
    nrows = int(np.ceil(n_items / ncols))
    cell_w = 1.0 / ncols
    cell_h = 1.0 / nrows
    # Force square cells: use the smaller dimension
    side = min(cell_w * (1 - 2 * pad_x), cell_h * (1 - 2 * pad_y))
    axes = []
    for i in range(n_items):
        row = i // ncols
        col = i % ncols
        # Center the square in each grid cell
        cx = col * cell_w + cell_w / 2
        cy = (nrows - 1 - row) * cell_h + cell_h / 2
        left = cx - side / 2
        bottom = cy - side / 2
        ax = ax_panel.inset_axes([left, bottom, side, side])
        ax.set_aspect('equal', adjustable='box')
        ax.set_aspect('equal', adjustable='box')
        axes.append(ax)
    return axes, ncols, nrows



In [ ]:
#| export
def plot_isoform_heatmap_composed(
    *,
    transcript_data,
    adata,
    gene_id: str,
    group_col: str,
    top_n: int = 2,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    visible_groups: Optional[List[str]] = None,
    cmap: str = "magma",
    colorbar_label: str = "PSI",
    fig_width: float = 18.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    add_gex_band: bool = False,
    add_group_color_band: bool = False,
    group_color_map: Optional[Dict[str, str]] = None,
    gex_band_position: str = "top",
    group_band_position: str = "bottom",
    show_panel_colorbar: bool = False,   # default off: use legend instead
    show_group_legend: bool = False,     # default off: group labels on x-axis are sufficient
    legend_pad: float = 0.02,            # horizontal padding between plot and legend
    label_offset_h: float = 0,  # horizontal offset for x-axis labels (positive = right)
    label_offset_v: float = 0,  # vertical offset for x-axis labels (positive = down)
    row_pitch: float = 0.45,
    intron_scale: float = 0.08,  # intron compression  -  matches standalone default
    color_offset: int = 0,  # rotate transcript color palette by this many steps
    palette: str = "wong",       # 'wong' | 'tol' | 'ghibli' | 'tab10'
    label_rot: int = 45,    # rotation for x-axis group labels
    font_scale: float = 1.0,  # scale all font sizes uniformly
    colors: Optional[Dict[str, str]] = None,  # isoform_id -> hex color overrides
    fig_height: float | None = None,  # force canvas height for alignment across plot types
    show_xlabels: bool = True,  # set False on top panels when stacking in Illustrator
    panel_width: float | None = None,  # fix data-panel width for cross-plot alignmentor
) -> Tuple[plt.Figure, List[str], List[str], np.ndarray]:
    """
    Plot composed figure with transcript structures and heatmap.

    Parameters
    ----------
    transcript_data : TranscriptData
        Transcript annotation data
    adata : AnnData
        Annotated data object with transcript counts
    gene_id : str
        Gene ID to plot
    group_col : str
        Column in adata.obs defining groups (e.g., 'cell_type')
    top_n : int, default 2
        Number of top isoforms to show
    estimator : str, default "pseudobulk"
        PSI estimator
    dirichlet_alpha : float, default 0.5
        Dirichlet alpha parameter
    epsilon : float, default 1e-6
        Small value to avoid division by zero
    visible_groups : List[str], optional
        Subset of groups to display
    cmap : str, default "magma"
        Colormap for PSI heatmap
    colorbar_label : str, default "PSI"
        Label for PSI colorbar
    fig_width : float, default 14.0
        Figure width in inches
    draw_cds : bool, default True
        Whether to draw CDS regions
    show_ticks : bool, default True
        Whether to show ruler ticks
    add_gex_band : bool, default False
        Whether to add gene expression band
    add_group_color_band : bool, default False
        Whether to add group color band
    group_color_map : Dict[str, str], optional
        Custom color map for groups
    gex_band_position : str, default "top"
        Position of GEX band ("top" or "bottom")
    group_band_position : str, default "bottom"
        Position of group color band ("top" or "bottom")
    show_panel_colorbar : bool, default False
        Whether to show colorbar on panel (vs legend)
    show_group_legend : bool, default False
        Whether to show categorical group legend (usually redundant with x-axis labels)
    legend_pad : float, default 0.02
        Horizontal padding between plot and legend colorbars
    label_offset_h : float, default 0
        Horizontal offset for x-axis labels (positive = right)
    label_offset_v : float, default 0
        Vertical offset for x-axis labels (positive = down)
    label_rot : int, default 35
        Rotation angle for x-axis group labels
    font_scale : float, default 1.0
        Multiply all font sizes by this factor (e.g. 1.3 for larger text)
    colors : Dict[str, str], optional
        Map isoform_id -> hex color string to override auto-assigned palette

    Returns
    -------
    fig : plt.Figure
        The matplotlib figure
    iso_ids : List[str]
        List of isoform IDs
    groups : List[str]
        List of group names
    V : np.ndarray
        PSI matrix (isoforms x groups)
    """
    _fs = lambda x: int(round(x * font_scale))
    # ---- compute PSI matrix ----
    result = _select_isoforms(
        adata, gene_id, group_col,
        top_n=top_n, estimator=estimator,
        dirichlet_alpha=dirichlet_alpha, epsilon=epsilon,
        visible_groups=visible_groups,
    )
    if result is None:
        return plt.figure(), [], [], np.zeros((0, 0), float)
    iso_ids, groups, V = result

    # ---- decide band placement / extra space ----
    want_bands = add_gex_band or add_group_color_band
    bands_frac_top, bands_frac_bottom, gpos, cpos = _compute_band_fracs(
        add_gex_band, add_group_color_band,
        gex_band_position, group_band_position, bands_frac=0.03,
    )

    # ---- build transcript + panel layout ----
    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
        color_offset=color_offset,
        palette=palette,
    )
    tp.row_pitch = row_pitch
    if colors is not None:
        tp.colors = [colors.get(tid) for tid in iso_ids]

    layout = make_aligned_transcript_panel_layout(
        tp,
        iso_ids,
        n_panel_cols=len(groups),
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=bands_frac_bottom,
        fig_height=fig_height,
        panel_width=panel_width,
    )

    # ---- main heatmap panel ----
    heat_meta = add_heatmap_to_layout(
        tp,
        layout,
        V,
        cmap=cmap,
        colorbar_label=colorbar_label,
        col_labels=groups,
        label_wrap=14,
        label_rot=label_rot,
        show_panel_colorbar=show_panel_colorbar,
    )
    # heat_meta must be a dict with vmin, vmax, cmap
    psi_norm = mpl.colors.Normalize(vmin=heat_meta["vmin"], vmax=heat_meta["vmax"])
    psi_cmap = mpl.cm.get_cmap(heat_meta["cmap"])
    # Reset axis limits to exact image extent for correct band alignment
    layout.ax_panel.set_xlim(-0.5, len(groups) - 0.5)
    layout.ax_panel.set_ylim(len(iso_ids) - 0.5, -0.5)

    # ----------------- optional bands -----------------
    cat_used, quant_used, group_color_map, quant_norm, quant_cmap = _populate_bands(
        layout, adata, gene_id, groups, group_col,
        add_gex_band=add_gex_band, add_group_color_band=add_group_color_band,
        group_color_map=group_color_map,
        gex_band_position=gex_band_position, group_band_position=group_band_position,
        epsilon=epsilon, tick_pad=35, font_scale=font_scale,
    )


    # Apply custom label offsets if specified
    if label_offset_h != 0.0 or label_offset_v != 0.0:
        import matplotlib.transforms as mtransforms
        ax = layout.ax_panel
        # Create an offset transform in points (horizontal and vertical)
        # For rotated labels, we offset in display coordinates (points)
        offset_h_points = label_offset_h * 72  # convert to points (assuming label_offset in inches)
        offset_v_points = label_offset_v * 72  # convert to points
        offset = mtransforms.ScaledTranslation(offset_h_points, -offset_v_points, ax.figure.dpi_scale_trans)
        
        for label in ax.get_xticklabels():
            # Apply offset to existing transform
            label.set_transform(label.get_transform() + offset)

    # ---- suppress x-axis labels for stacked Illustrator layout ----
    if not show_xlabels:
        layout.ax_panel.set_xticklabels([])
        layout.ax_panel.tick_params(axis='x', which='both', length=0)

    # ---- legend panel: unified builder ----
    builder = LegendPanelBuilder(getattr(layout, "ax_legend", None))

    if show_group_legend and cat_used and group_color_map is not None:
        builder.add_categorical(groups, group_color_map, title="Groups")

    if quant_used and quant_norm is not None and quant_cmap is not None:
        builder.add_colorbar(quant_norm, quant_cmap, label="GEX\nCP10k")

    if psi_norm is not None and psi_cmap is not None:
        builder.add_colorbar(psi_norm, psi_cmap, label=colorbar_label)

    if builder._slots:
        builder.render()
    else:
        builder.hide()

    layout.fig.canvas.draw()
    for _ax in layout.fig.axes:
        if _ax.get_axes_locator() is not None:
            _ax.set_position(_ax.get_position())
            _ax.set_axes_locator(None)
    return layout.fig, iso_ids, groups, V

In [ ]:
#| export  
def plot_isoform_dot_composed(
    *,
    transcript_data,
    adata,
    gene_id: str,
    group_col: str,
    top_n: int = 2,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    visible_groups: Optional[List[str]] = None,
    size_mode: str = "prevalence",
    psi_threshold: float = 0.05,
    cmap: str = "magma",
    colorbar_label: str = "PSI",
    fig_width: float = 18.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    add_gex_band: bool = False,
    add_group_color_band: bool = False,
    group_color_map: Optional[Dict[str, str]] = None,
    gex_band_position: str = "top",
    group_band_position: str = "bottom",
    show_panel_colorbar: bool = False,   # use legend instead
    show_group_legend: bool = False,     # default off: group labels on x-axis are sufficient
    label_offset_h: float = 0.0,  # horizontal offset for x-axis labels (positive = right)
    label_offset_v: float = 0.0,  # vertical offset for x-axis labels (positive = down)
    row_pitch: float = 0.45,
    intron_scale: float = 0.08,  # intron compression (smaller = more compressed)
    color_offset: int = 0,  # rotate transcript color palette by this many steps
) -> Tuple[plt.Figure, List[str], List[str], np.ndarray, np.ndarray]:

    # ---------- compute matrices ----------
    result = _select_isoforms_dot(
        adata, gene_id, group_col,
        top_n=top_n, estimator=estimator,
        dirichlet_alpha=dirichlet_alpha, epsilon=epsilon,
        visible_groups=visible_groups,
        size_mode=size_mode, psi_threshold=psi_threshold,
    )
    if result is None:
        return plt.figure(), [], [], np.zeros((0, 0)), np.zeros((0, 0))
    iso_ids, groups, V, S = result

    # ---------- layout / bands ----------
    want_bands = add_gex_band or add_group_color_band
    bands_frac_top, bands_frac_bottom, gpos, cpos = _compute_band_fracs(
        add_gex_band, add_group_color_band,
        gex_band_position, group_band_position, bands_frac=0.03,
    )

    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
        color_offset=color_offset,
    )
    tp.row_pitch = row_pitch

    layout = make_aligned_transcript_panel_layout(
        tp,
        iso_ids,
        n_panel_cols=len(groups),
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=bands_frac_bottom,
    )

    # ---------- main dot panel ----------
    dot_meta = add_dotpanel_to_layout(
        tp,
        layout,
        dot_color=V,
        dot_size=S,
        cmap=cmap,
        colorbar_label=colorbar_label,
        col_labels=groups,
        label_wrap=14,
        label_rot=45,
        show_panel_colorbar=show_panel_colorbar,
    )
    psi_norm = mpl.colors.Normalize(vmin=dot_meta["vmin"], vmax=dot_meta["vmax"])
    psi_cmap = mpl.cm.get_cmap(dot_meta["cmap"])

    # ---------- optional top/bottom bands (GEX strip, categorical strip) ----------
    cat_used, quant_used, group_color_map, quant_norm, quant_cmap = _populate_bands(
        layout, adata, gene_id, groups, group_col,
        add_gex_band=add_gex_band, add_group_color_band=add_group_color_band,
        group_color_map=group_color_map,
        gex_band_position=gex_band_position, group_band_position=group_band_position,
        epsilon=epsilon,
    )


    # Apply custom label offsets if specified
    if label_offset_h != 0.0 or label_offset_v != 0.0:
        import matplotlib.transforms as mtransforms
        ax = layout.ax_panel
        # Create an offset transform in points
        offset_h_points = label_offset_h * 72
        offset_v_points = label_offset_v * 72
        offset = mtransforms.ScaledTranslation(offset_h_points, -offset_v_points, ax.figure.dpi_scale_trans)
        
        for label in ax.get_xticklabels():
            label.set_transform(label.get_transform() + offset)
    # ---------- legend axis on the far right ----------
    # ---------- legend axis on the far right ----------
    builder = LegendPanelBuilder(getattr(layout, "ax_legend", None))
    
    if show_group_legend and cat_used and group_color_map is not None:
        builder.add_categorical(groups, group_color_map, title="Groups")
    
    if psi_norm is not None and psi_cmap is not None:
        builder.add_colorbar(psi_norm, psi_cmap, label="PSI")
    
    if quant_used and quant_norm is not None and quant_cmap is not None:
        builder.add_colorbar(quant_norm, quant_cmap, label="GEX\nCP10k")
    
    if S.size > 0:
        builder.add_dot_size_legend()
    
    if builder._slots:
        builder.render()
    else:
        builder.hide()

    return layout.fig, iso_ids, groups, V, S

## Examples
> Composed plots combining transcript structures with quantitative visualizations

In [ ]:
#| export
def plot_isoform_density_composed(
    *,
    transcript_data,
    adata,
    gene_id: str,
    group_col: str,
    top_n: int = 2,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    visible_groups: Optional[List[str]] = None,
    show_individual_colorbars: bool = False,
    use_per_panel_scale: bool = False,
    fig_width: float = 18.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    add_gex_band: bool = False,
    add_group_color_band: bool = False,
    group_color_map: Optional[Dict[str, str]] = None,
    density_basis: str = "umap",
    density_cmaps: Optional[List[str]] = None,
    density_max_cols: int = 2,
    density_size: float = 5.0,
    density_alpha: float = 0.9,
    density_adjust: float = 1.0,
    use_global_vmin_vmax: bool = True,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    row_pitch: float = 0.45,  # vertical spacing between transcript rows
    intron_scale: float = 0.15,  # intron compression (smaller = more compressed)
) -> Tuple[plt.Figure, List[str], List[str]]:
    """
    Composed plot: transcript structures + density (KDE) embeddings.
    
    Left: Transcript structure glyphs
    Right: Tiled density plots (one per isoform, showing spatial density via KDE)
    
    Parameters
    ----------
    show_individual_colorbars : bool
        If True, show a small colorbar for each density plot panel.
    use_per_panel_scale : bool
        If True, each panel has its own colorbar scale based on its data.
        If False (default), all panels share a global scale.
    """
    # 1) Get isoforms and groups
    iso_ids, groups, V = _compute_group_matrix_from_adata(
        adata, gene_id, group_col,
        top_n=top_n, estimator=estimator,
        dirichlet_alpha=dirichlet_alpha, epsilon=epsilon
    )
    
    if not iso_ids or V.size == 0:
        return plt.figure(), [], []
    
    if visible_groups is not None:
        keep = [groups.index(g) for g in visible_groups if g in groups]
        if not keep:
            return plt.figure(), [], []
        groups = [groups[i] for i in keep]
    
    # 2) Bands
    want_bands = add_gex_band or add_group_color_band
    bands_frac_top = 0.10 if want_bands else 0.0
    
    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
    )
    tp.row_pitch = row_pitch
    
    layout = make_transcript_panel_layout(
        tp, iso_ids,
        n_panel_cols=len(groups),
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=0.0,
    )
    fig = layout.fig
    
    # Hide colorbar axis
    if layout.ax_cbar is not None:
        layout.ax_cbar.set_visible(False)
        layout.ax_cbar.axis("off")
    
    # 3) Get coordinates
    coords = get_coords_from_adata(adata, basis=density_basis)
    
    # 4) Calculate density for each isoform
    transcripts = list(iso_ids)
    n_iso = len(transcripts)
    if n_iso == 0:
        return fig, iso_ids, groups
    
    # Calculate density values for each isoform
    density_values = []
    for tr in transcripts:
        density = calculate_density(
            adata,
            feature=tr,
            basis=density_basis,
            adjust=density_adjust,
            map_to_cells=True,
        )
        density_values.append(density)
    
    density_values = np.array(density_values)  # (n_iso, n_cells)
    
    # Global vmin/vmax for density (only computed if not using per-panel scale)
    if not use_per_panel_scale and use_global_vmin_vmax:
        if vmin is None:
            vmin = float(np.nanpercentile(density_values, 1.0))
        if vmax is None:
            vmax = float(np.nanpercentile(density_values, 99.0))
    
    if density_cmaps is None:
        density_cmaps = ["Reds"] * n_iso
    elif len(density_cmaps) != n_iso:
        raise ValueError("density_cmaps must be None or same length as iso_ids")
    
    # 5) Tile density plots inside panel
    ax_panel = layout.ax_panel
    ax_panel.set_axis_off()
    
    ncols = min(density_max_cols, n_iso)
    nrows = int(np.ceil(n_iso / ncols))
    
    cell_w = 1.0 / ncols
    cell_h = 1.0 / nrows
    
    pad_x_frac = 0.15
    pad_y_frac = 0.10
    
    mappable_for_cbar = None  # for shared colorbar
    
    for i, (tr, cmap, vals) in enumerate(zip(transcripts, density_cmaps, density_values)):
        row = i // ncols
        col = i % ncols
        
        base_left = col * cell_w
        base_bottom = (nrows - 1 - row) * cell_h
        
        left = base_left + pad_x_frac * cell_w
        bottom = base_bottom + pad_y_frac * cell_h
        width = cell_w * (1 - 2 * pad_x_frac)
        height = cell_h * (1 - 2 * pad_y_frac)
        
        ax = ax_panel.inset_axes([left, bottom, width, height])
        
        # Compute per-panel scale if requested
        if use_per_panel_scale:
            panel_vmin = float(np.nanpercentile(vals, 1.0))
            panel_vmax = float(np.nanpercentile(vals, 99.0))
        else:
            panel_vmin = vmin
            panel_vmax = vmax
        
        sc = ax.scatter(
            coords[:, 0], coords[:, 1],
            c=vals, s=density_size, cmap=cmap,
            alpha=density_alpha, vmin=panel_vmin, vmax=panel_vmax,
            linewidths=0, rasterized=True,
        )
        
        # Remember mappable for shared colorbar
        if not use_per_panel_scale:
            mappable_for_cbar = sc
        
        # Strip version suffix and add "Density" label
        title_str = tr.split(".")[0] + " (Density)"
        ax.set_title(title_str, fontsize=9)
        
        # Only bottom row / left column get axis labels
        show_xlab = (row == nrows - 1)
        show_ylab = (col == 0)
        
        # Set axis labels based on basis
        if density_basis.lower() == "umap":
            xlabel, ylabel = "UMAP1", "UMAP2"
        elif density_basis.lower() == "tsne":
            xlabel, ylabel = "tSNE1", "tSNE2"
        else:
            xlabel, ylabel = f"{density_basis}1", f"{density_basis}2"
        
        if show_xlab:
            ax.set_xlabel(xlabel, fontsize=8)
        else:
            ax.set_xlabel("")
        if show_ylab:
            ax.set_ylabel(ylabel, fontsize=8)
        else:
            ax.set_ylabel("")
        
        ax.tick_params(axis="x", labelbottom=show_xlab, bottom=True, length=2, width=0.5)
        ax.tick_params(axis="y", labelleft=show_ylab, left=True, length=2, width=0.5)
        
        # Per-panel colorbar using inset_axes (only if per-panel scale)
        if show_individual_colorbars and use_per_panel_scale:
            cax = ax.inset_axes([1.02, 0.1, 0.04, 0.8])  # [x0, y0, w, h] in axes coords
            cb = fig.colorbar(sc, cax=cax)
            cb.ax.tick_params(labelsize=6)
    
    # Shared colorbar (only if using global scaling and colorbars requested)
    if show_individual_colorbars and not use_per_panel_scale and mappable_for_cbar is not None:
        bbox = ax_panel.get_position(fig)
        cbar_width = 0.012
        cbar_pad = 0.01
        cax = fig.add_axes([
            bbox.x1 + cbar_pad,
            bbox.y0,
            cbar_width,
            bbox.height,
        ])
        cb = fig.colorbar(mappable_for_cbar, cax=cax)
        cb.set_label("Density", fontsize=8)
        cb.ax.tick_params(labelsize=8)
    
    # 6) Optional bands
    if want_bands and layout.ax_band_top is not None and len(groups) > 0:
        n_slots = int(add_group_color_band) + int(add_gex_band)
        slot = 0
        
        if add_group_color_band:
            if group_color_map is None:
                cmap_cat = mpl.cm.get_cmap("tab20")
                group_color_map = {
                    g: mpl.colors.to_hex(cmap_cat(i / max(1, len(groups))))
                    for i, g in enumerate(groups)
                }
            add_categorical_band_to_layout(
                layout, groups, color_map=group_color_map,
                slot=slot, n_slots=n_slots, which="top"
            )
            slot += 1
        
        if add_gex_band:
            X_all = _dense_X(adata.X)
            if "geneId" in adata.var:
                m = adata.var["geneId"].astype(str).values == str(gene_id)
            else:
                m = np.zeros(adata.n_vars, bool)
            
            if not np.any(m):
                vals_band = np.zeros(len(groups), float)
            else:
                Xg = X_all[:, m]
                if Xg.ndim == 2 and Xg.shape[1] > 1:
                    Xg = Xg.sum(1)
                lib_tot = X_all.sum(1) + epsilon
                
                all_groups = adata.obs[group_col].astype(str).values
                vals_band = []
                for g in groups:
                    sel = (all_groups == g)
                    if not sel.any():
                        vals_band.append(0.0)
                        continue
                    cp10k = (Xg[sel] / lib_tot[sel]) * 1e4
                    vals_band.append(float(np.nanmean(cp10k)))
                vals_band = np.asarray(vals_band, float)
            
            add_quant_band_to_layout(
                layout, vals_band, cmap="viridis",
                slot=slot, n_slots=n_slots, which="top"
            )
    
    return fig, iso_ids, groups

In [ ]:
#| export
def plot_isoform_spatial_composed(
    *,
    transcript_data,
    adata,
    gene_id: str,
    group_col: str,
    top_n: int = 2,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    visible_groups: Optional[List[str]] = None,
    show_individual_colorbars: bool = True,
    use_per_panel_scale: bool = False,
    fig_width: float = 28.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    add_gex_band: bool = False,
    add_group_color_band: bool = False,
    group_color_map: Optional[Dict[str, str]] = None,
    spatial_basis: str = "spatial",
    spatial_cmap: Optional[str] = None,
    spatial_cmaps: Optional[List[str]] = None,
    spatial_max_cols: int = 2,
    spatial_size: float = 5.0,
    spatial_alpha: float = 0.9,
    use_global_vmin_vmax: bool = True,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    invert_y: bool = True,
    use_density: bool = False,
    density_adjust: float = 1.0,
    aspect_equal: bool = True,
    color_offset: int = 0,
    intron_scale: float = 0.08,  # intron compression (smaller = more compressed)
) -> Tuple[plt.Figure, List[str], List[str]]:
    """
    Composed plot: transcript structures + spatial embeddings.
    
    Left: Transcript structure glyphs
    Right: Tiled spatial plots (one per isoform)
    
    If use_density=False (default): colour = raw isoform expression.
    If use_density=True: colour = KDE-based spatial density of that isoform.

    spatial_cmap: single colormap name applied to all panels (e.g. "Reds", "magma").
        Overrides the per-mode default. spatial_cmaps takes precedence if also provided.
    spatial_cmaps: per-isoform colormap list; overrides spatial_cmap.
    aspect_equal: if True (default), each spatial subplot uses equal x/y scaling,
        preserving true tissue geometry. Set False to fill the available cell area.
    use_per_panel_scale: if True, each panel has its own colorbar scale based on its data.
        If False (default), all panels share a global scale.
    color_offset: rotate transcript color palette by this many steps.
    """
    # 1) Get isoforms and groups
    iso_ids, groups, V = _compute_group_matrix_from_adata(
        adata, gene_id, group_col,
        top_n=top_n, estimator=estimator,
        dirichlet_alpha=dirichlet_alpha, epsilon=epsilon
    )
    
    if not iso_ids or V.size == 0:
        return plt.figure(), [], []
    
    if visible_groups is not None:
        keep = [groups.index(g) for g in visible_groups if g in groups]
        if not keep:
            return plt.figure(), [], []
        groups = [groups[i] for i in keep]
    
    # 2) Bands
    want_bands = add_gex_band or add_group_color_band
    bands_frac_top = 0.10 if want_bands else 0.0
    
    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
        color_offset=color_offset,
    )
    
    layout = make_transcript_panel_layout(
        tp, iso_ids,
        n_panel_cols=len(groups),
        panel_width=0.85,
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=0.0,
    )
    fig = layout.fig
    
    # Hide colorbar axis
    if layout.ax_cbar is not None:
        layout.ax_cbar.set_visible(False)
        layout.ax_cbar.axis("off")
    
    # 3) Get spatial coordinates
    coords = get_coords_from_adata(adata, basis=spatial_basis)
    
    # 4) Get values to plot (expression or density)
    transcripts = list(iso_ids)
    n_iso = len(transcripts)
    if n_iso == 0:
        return fig, iso_ids, groups
    
    if use_density:
        # Calculate density for each isoform
        value_list = []
        for tr in transcripts:
            density = calculate_density(
                adata,
                feature=tr,
                basis=spatial_basis,
                adjust=density_adjust,
                map_to_cells=True,
            )
            value_list.append(density)
        expr = np.array(value_list)  # (n_iso, n_cells)
        _default_cmap = "Reds"
    else:
        # Get expression values
        X_sub = adata[:, transcripts].X
        if issparse(X_sub):
            X_sub = X_sub.toarray()
        expr = X_sub.T  # (n_iso, n_cells)
        _default_cmap = "viridis"
    
    # Resolve colormap list: spatial_cmaps > spatial_cmap > per-mode default
    if spatial_cmaps is None:
        spatial_cmaps = [spatial_cmap or _default_cmap] * n_iso
    
    # Global vmin/vmax (only computed if not using per-panel scale)
    if not use_per_panel_scale and use_global_vmin_vmax:
        if vmin is None:
            vmin = float(np.nanpercentile(expr, 1.0))
        if vmax is None:
            vmax = float(np.nanpercentile(expr, 99.0))
    
    if len(spatial_cmaps) != n_iso:
        raise ValueError("spatial_cmaps must be None or same length as iso_ids")
    
    # 5) Tile spatial plots inside panel
    ax_panel = layout.ax_panel
    ax_panel.set_axis_off()
    
    ncols = min(spatial_max_cols, n_iso)
    nrows = int(np.ceil(n_iso / ncols))
    
    cell_w = 1.0 / ncols
    cell_h = 1.0 / nrows
    
    pad_x_frac = 0.15
    pad_y_frac = 0.10
    
    mappable_for_cbar = None  # for shared colorbar
    
    for i, (tr, cmap, vals) in enumerate(zip(transcripts, spatial_cmaps, expr)):
        row = i // ncols
        col = i % ncols
        
        base_left = col * cell_w
        base_bottom = (nrows - 1 - row) * cell_h
        
        left = base_left + pad_x_frac * cell_w
        bottom = base_bottom + pad_y_frac * cell_h
        width = cell_w * (1 - 2 * pad_x_frac)
        height = cell_h * (1 - 2 * pad_y_frac)
        
        ax = ax_panel.inset_axes([left, bottom, width, height])
        ax.set_aspect('equal', adjustable='box')
        
        # Compute per-panel scale if requested
        if use_per_panel_scale:
            panel_vmin = float(np.nanpercentile(vals, 1.0))
            panel_vmax = float(np.nanpercentile(vals, 99.0))
        else:
            panel_vmin = vmin
            panel_vmax = vmax
        
        sc = ax.scatter(
            coords[:, 0], coords[:, 1],
            c=vals, s=spatial_size, cmap=cmap,
            alpha=spatial_alpha, vmin=panel_vmin, vmax=panel_vmax,
            linewidths=0, rasterized=True,
        )
        
        # Remember mappable for shared colorbar
        if not use_per_panel_scale:
            mappable_for_cbar = sc
        
        # Invert y-axis if requested
        if invert_y:
            ax.invert_yaxis()
        
        # Preserve true tissue geometry (avoids distortion of spatial coordinates)
        if aspect_equal:
            ax.set_aspect('equal', adjustable='box')
        
        # Strip version suffix and add "(Density)" if needed
        title_str = tr.split(".")[0]
        if use_density:
            title_str += " (Density)"
        ax.set_title(title_str, fontsize=9)
        
        # Only bottom row / left column get axis labels
        show_xlab = (row == nrows - 1)
        show_ylab = (col == 0)
        
        if show_xlab:
            ax.set_xlabel("X", fontsize=8)
        else:
            ax.set_xlabel("")
        if show_ylab:
            ax.set_ylabel("Y", fontsize=8)
        else:
            ax.set_ylabel("")
        
        ax.tick_params(axis="x", labelbottom=show_xlab, bottom=True, length=2, width=0.5)
        ax.tick_params(axis="y", labelleft=show_ylab, left=True, length=2, width=0.5)
        
        # Per-panel colorbar (only if per-panel scale)
        if show_individual_colorbars and use_per_panel_scale:
            cax = ax.inset_axes([1.02, 0.1, 0.04, 0.8])
            cb = fig.colorbar(sc, cax=cax)
            cb.ax.tick_params(labelsize=6)
    
    # Shared colorbar (only if using global scaling and colorbars requested)
    if show_individual_colorbars and not use_per_panel_scale and mappable_for_cbar is not None:
        bbox = ax_panel.get_position(fig)
        cbar_width = 0.012
        cbar_pad = 0.01
        cax = fig.add_axes([
            bbox.x1 + cbar_pad,
            bbox.y0,
            cbar_width,
            bbox.height,
        ])
        cb = fig.colorbar(mappable_for_cbar, cax=cax)
        cb.ax.tick_params(labelsize=7)
    
    # 6) Optional bands
    if want_bands and layout.ax_band_top is not None and len(groups) > 0:
        n_slots = int(add_group_color_band) + int(add_gex_band)
        slot = 0
        
        if add_group_color_band:
            if group_color_map is None:
                cmap_cat = mpl.cm.get_cmap("tab20")
                group_color_map = {
                    g: mpl.colors.to_hex(cmap_cat(i / max(1, len(groups))))
                    for i, g in enumerate(groups)
                }
            add_categorical_band_to_layout(
                layout, groups, color_map=group_color_map,
                slot=slot, n_slots=n_slots, which="top"
            )
            slot += 1
        
        if add_gex_band:
            X_all = _dense_X(adata.X)
            if "geneId" in adata.var:
                m = adata.var["geneId"].astype(str).values == str(gene_id)
            else:
                m = np.zeros(adata.n_vars, bool)
            
            if not np.any(m):
                vals_band = np.zeros(len(groups), float)
            else:
                Xg = X_all[:, m]
                if Xg.ndim == 2 and Xg.shape[1] > 1:
                    Xg = Xg.sum(1)
                lib_tot = X_all.sum(1) + epsilon
                
                all_groups = adata.obs[group_col].astype(str).values
                vals_band = []
                for g in groups:
                    sel = (all_groups == g)
                    if not sel.any():
                        vals_band.append(0.0)
                        continue
                    cp10k = (Xg[sel] / lib_tot[sel]) * 1e4
                    vals_band.append(float(np.nanmean(cp10k)))
                vals_band = np.asarray(vals_band, float)
            
            add_quant_band_to_layout(
                layout, vals_band, cmap="viridis",
                slot=slot, n_slots=n_slots, which="top"
            )
    
    return fig, iso_ids, groups

In [ ]:
#| export
def plot_isoform_umap_composed(
    *,
    transcript_data,
    adata,
    gene_id: str,
    group_col: str,
    top_n: int = 2,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    visible_groups: Optional[List[str]] = None,
    show_individual_colorbars: bool = False,
    use_per_panel_scale: bool = False,
    fig_width: float = 18.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    add_gex_band: bool = False,
    add_group_color_band: bool = False,
    group_color_map: Optional[Dict[str, str]] = None,
    umap_basis: str = "umap",
    umap_cmaps: Optional[List[str]] = None,
    umap_max_cols: int = 2,
    umap_size: float = 5.0,
    umap_alpha: float = 0.9,
    use_global_vmin_vmax: bool = True,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    use_density: bool = False,
    density_adjust: float = 1.0,
    row_pitch: float = 0.45,  # vertical spacing between transcript rows
    intron_scale: float = 0.15,  # intron compression (smaller = more compressed)
) -> Tuple[plt.Figure, List[str], List[str]]:
    """
    Composed plot: transcript structures + UMAP embeddings.
    
    Left: Transcript structure glyphs
    Right: Tiled UMAP plots (one per isoform)
    
    Parameters
    ----------
    show_individual_colorbars : bool
        If True, show a small colorbar for each UMAP plot panel.
    use_per_panel_scale : bool
        If True, each panel has its own colorbar scale based on its data.
        If False (default), all panels share a global scale.
    use_density : bool
        If True, show KDE-based density instead of raw expression.
    density_adjust : float
        Bandwidth adjustment for KDE (only used if use_density=True).
    """
    # 1) Get isoforms and groups
    iso_ids, groups, V = _compute_group_matrix_from_adata(
        adata, gene_id, group_col,
        top_n=top_n, estimator=estimator,
        dirichlet_alpha=dirichlet_alpha, epsilon=epsilon
    )
    
    if not iso_ids or V.size == 0:
        return plt.figure(), [], []
    
    if visible_groups is not None:
        keep = [groups.index(g) for g in visible_groups if g in groups]
        if not keep:
            return plt.figure(), [], []
        groups = [groups[i] for i in keep]
    
    # 2) Bands
    want_bands = add_gex_band or add_group_color_band
    bands_frac_top = 0.10 if want_bands else 0.0
    
    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
    )
    tp.row_pitch = row_pitch
    
    layout = make_transcript_panel_layout(
        tp, iso_ids,
        n_panel_cols=len(groups),
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=0.0,
    )
    fig = layout.fig
    
    # Hide colorbar axis
    if layout.ax_cbar is not None:
        layout.ax_cbar.set_visible(False)
        layout.ax_cbar.axis("off")
    
    # 3) Get UMAP coordinates
    coord_key = f"X_{umap_basis}"
    if coord_key not in adata.obsm:
        raise ValueError(f"AnnData does not contain .obsm['{coord_key}'].")
    
    coords = np.asarray(adata.obsm[coord_key])[:, :2]
    
    # 4) Get values to plot (expression or density)
    transcripts = list(iso_ids)
    n_iso = len(transcripts)
    if n_iso == 0:
        return fig, iso_ids, groups
    
    if use_density:
        # Calculate density for each isoform
        value_list = []
        for tr in transcripts:
            density = calculate_density(
                adata,
                feature=tr,
                basis=umap_basis,
                adjust=density_adjust,
                map_to_cells=True,
            )
            value_list.append(density)
        expr = np.array(value_list)  # (n_iso, n_cells)
        
        # Default colormap for density
        if umap_cmaps is None:
            umap_cmaps = ["Reds"] * n_iso
    else:
        # Get expression values
        X_sub = adata[:, transcripts].X
        if issparse(X_sub):
            X_sub = X_sub.toarray()
        expr = X_sub.T  # (n_iso, n_cells)
        
        # Default colormap for expression
        if umap_cmaps is None:
            umap_cmaps = ["viridis"] * n_iso
    
    # Global vmin/vmax (only computed if not using per-panel scale)
    if not use_per_panel_scale and use_global_vmin_vmax:
        if vmin is None:
            vmin = float(np.nanpercentile(expr, 1.0))
        if vmax is None:
            vmax = float(np.nanpercentile(expr, 99.0))
    
    if len(umap_cmaps) != n_iso:
        raise ValueError("umap_cmaps must be None or same length as iso_ids")
    
    # 5) Tile UMAP plots inside panel
    ax_panel = layout.ax_panel
    ax_panel.set_axis_off()
    
    ncols = min(umap_max_cols, n_iso)
    nrows = int(np.ceil(n_iso / ncols))
    
    cell_w = 1.0 / ncols
    cell_h = 1.0 / nrows
    
    pad_x_frac = 0.15
    pad_y_frac = 0.10
    
    mappable_for_cbar = None  # for shared colorbar
    
    for i, (tr, cmap, vals) in enumerate(zip(transcripts, umap_cmaps, expr)):
        row = i // ncols
        col = i % ncols
        
        base_left = col * cell_w
        base_bottom = (nrows - 1 - row) * cell_h
        
        left = base_left + pad_x_frac * cell_w
        bottom = base_bottom + pad_y_frac * cell_h
        width = cell_w * (1 - 2 * pad_x_frac)
        height = cell_h * (1 - 2 * pad_y_frac)
        
        ax = ax_panel.inset_axes([left, bottom, width, height])
        
        # Compute per-panel scale if requested
        if use_per_panel_scale:
            panel_vmin = float(np.nanpercentile(vals, 1.0))
            panel_vmax = float(np.nanpercentile(vals, 99.0))
        else:
            panel_vmin = vmin
            panel_vmax = vmax
        
        sc = ax.scatter(
            coords[:, 0], coords[:, 1],
            c=vals, s=umap_size, cmap=cmap,
            alpha=umap_alpha, vmin=panel_vmin, vmax=panel_vmax,
            linewidths=0, rasterized=True,
        )
        
        # Remember mappable for shared colorbar
        if not use_per_panel_scale:
            mappable_for_cbar = sc
        
        # Strip version suffix and add "(Density)" if needed
        title_str = tr.split(".")[0]
        if use_density:
            title_str += " (Density)"
        ax.set_title(title_str, fontsize=9)
        
        # Only bottom row / left column get axis labels
        show_xlab = (row == nrows - 1)
        show_ylab = (col == 0)
        
        if show_xlab:
            ax.set_xlabel("UMAP1", fontsize=8)
        else:
            ax.set_xlabel("")
        if show_ylab:
            ax.set_ylabel("UMAP2", fontsize=8)
        else:
            ax.set_ylabel("")
        
        ax.tick_params(axis="x", labelbottom=show_xlab, bottom=True, length=2, width=0.5)
        ax.tick_params(axis="y", labelleft=show_ylab, left=True, length=2, width=0.5)
        
        # Per-panel colorbar using inset_axes (only if per-panel scale)
        if show_individual_colorbars and use_per_panel_scale:
            cax = ax.inset_axes([1.02, 0.1, 0.04, 0.8])  # [x0, y0, w, h] in axes coords
            cb = fig.colorbar(sc, cax=cax)
            cb.ax.tick_params(labelsize=6)
    
    # Shared colorbar (only if using global scaling and colorbars requested)
    if show_individual_colorbars and not use_per_panel_scale and mappable_for_cbar is not None:
        bbox = ax_panel.get_position(fig)
        cbar_width = 0.012
        cbar_pad = 0.01
        cax = fig.add_axes([
            bbox.x1 + cbar_pad,
            bbox.y0,
            cbar_width,
            bbox.height,
        ])
        cb = fig.colorbar(mappable_for_cbar, cax=cax)
        cb.set_label("Density" if use_density else "Expression", fontsize=8)
        cb.ax.tick_params(labelsize=8)
    
    # 6) Optional bands
    if want_bands and layout.ax_band_top is not None and len(groups) > 0:
        n_slots = int(add_group_color_band) + int(add_gex_band)
        slot = 0
        
        if add_group_color_band:
            if group_color_map is None:
                cmap_cat = mpl.cm.get_cmap("tab20")
                group_color_map = {
                    g: mpl.colors.to_hex(cmap_cat(i / max(1, len(groups))))
                    for i, g in enumerate(groups)
                }
            add_categorical_band_to_layout(
                layout, groups, color_map=group_color_map,
                slot=slot, n_slots=n_slots, which="top"
            )
            slot += 1
        
        if add_gex_band:
            X_all = _dense_X(adata.X)
            if "geneId" in adata.var:
                m = adata.var["geneId"].astype(str).values == str(gene_id)
            else:
                m = np.zeros(adata.n_vars, bool)
            
            if not np.any(m):
                vals_band = np.zeros(len(groups), float)
            else:
                Xg = X_all[:, m]
                if Xg.ndim == 2 and Xg.shape[1] > 1:
                    Xg = Xg.sum(1)
                lib_tot = X_all.sum(1) + epsilon
                
                all_groups = adata.obs[group_col].astype(str).values
                vals_band = []
                for g in groups:
                    sel = (all_groups == g)
                    if not sel.any():
                        vals_band.append(0.0)
                        continue
                    cp10k = (Xg[sel] / lib_tot[sel]) * 1e4
                    vals_band.append(float(np.nanmean(cp10k)))
                vals_band = np.asarray(vals_band, float)
            
            add_quant_band_to_layout(
                layout, vals_band, cmap="viridis",
                slot=slot, n_slots=n_slots, which="top"
            )
    
    return fig, iso_ids, groups

In [ ]:
#| export
def plot_isoform_replicates_composed(
    *,
    transcript_data,
    adata,
    gene_id: str,
    group_col: str,
    replicate_col: str,
    top_n: int = 2,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    visible_groups: Optional[List[str]] = None,
    fig_width: float = 18.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    add_gex_band: bool = False,
    add_group_color_band: bool = False,
    group_color_map: Optional[Dict[str, str]] = None,
    box_overlay: bool = True,
    point_size: float = 7.0,
    point_alpha: float = 0.85,
    jitter_width: float = 0.12,
    label_wrap: int = 14,
    label_rot: int = 45,
    row_pitch: float = 0.45,
    intron_scale: float = 0.08,  # intron compression  -  matches standalone default
    color_offset: int = 0,  # rotate transcript color palette by this many steps
    palette: str = "wong",       # 'wong' | 'tol' | 'ghibli' | 'tab10'
    font_scale: float = 1.0,  # scale all font sizes uniformly
    colors: Optional[Dict[str, str]] = None,  # isoform_id -> hex color overrides
    show_xlabel: bool = False,  # show group_col as x-axis label on bottom subplot
    fig_height: float | None = None,  # force canvas height for alignment across plot types
    show_xlabels: bool = True,  # set False on top panels when stacking in Illustrator
    panel_width: float | None = None,  # fix data-panel width for cross-plot alignmentor
) -> Tuple[plt.Figure, List[str], List[str], Dict[Tuple[int, int], np.ndarray]]:
    """
    Composed plot: transcript structures + replicate PSI variability.
    
    Left: Transcript structure glyphs
    Right: Replicate plots showing PSI variability across biological replicates
    
    Parameters
    ----------
    transcript_data : TranscriptData
        Transcript data object for structure plots
    adata : AnnData
        Annotated data object
    gene_id : str
        Gene ID to plot
    group_col : str
        Column in adata.obs defining groups (e.g., 'cell_type')
    replicate_col : str
        Column in adata.obs defining replicates (e.g., 'batch', 'sample_id')
    top_n : int
        Number of top isoforms to plot
    estimator : str
        PSI estimator ('pseudobulk', 'dirichlet', 'cell-mean', 'cell-median', 'coverage-weighted')
    dirichlet_alpha : float
        Dirichlet alpha parameter
    epsilon : float
        Small value to avoid division by zero
    visible_groups : List[str], optional
        Subset of groups to display
    fig_width : float
        Figure width in inches
    draw_cds : bool
        If True, draw CDS regions in transcript structures
    show_ticks : bool
        If True, show ruler ticks
    add_gex_band : bool
        If True, add GEX band
    add_group_color_band : bool
        If True, add group color band
    group_color_map : Dict[str, str], optional
        Custom color map for groups
    box_overlay : bool
        If True, overlay boxplots on scatter points
    point_size : float
        Size of scatter points
    point_alpha : float
        Alpha transparency of points
    jitter_width : float
        Width of horizontal jitter for points
    label_wrap : int
        Width for wrapping x-axis labels
    label_rot : int
        Rotation angle for x-axis labels
    color_offset : int
        Rotate transcript color palette by this many steps
    font_scale : float, default 1.0
        Multiply all font sizes by this factor
    colors : Dict[str, str], optional
        Map isoform_id -> hex color string to override auto-assigned palette
    show_xlabel : bool, default False
        Show group_col name as x-axis label on the bottom subplot
        
    Returns
    -------
    fig : plt.Figure
        The matplotlib figure
    iso_ids : List[str]
        List of isoform IDs
    groups : List[str]
        List of group names
    replicate_data : Dict[Tuple[int, int], np.ndarray]
        Dictionary of replicate PSI values
        
    Examples
    --------
    # Replicate plot showing biological variability
    fig, iso_ids, groups, data = plot_isoform_replicates_composed(
        transcript_data=td,
        adata=adata,
        gene_id="Myl6",
        group_col="cell_type",
        replicate_col="batch",
        top_n=3,
        fig_width=14.0,
    )
    plt.show()
    """
    _fs = lambda x: int(round(x * font_scale))
    # 1) Compute replicate PSI data
    iso_ids, groups, replicate_data = _compute_replicate_psi_from_adata(
        adata,
        gene_id=gene_id,
        group_col=group_col,
        replicate_col=replicate_col,
        top_n=top_n,
        epsilon=epsilon,
        estimator=estimator,
        dirichlet_alpha=dirichlet_alpha,
    )
    
    if not iso_ids:
        return plt.figure(), [], [], {}
    
    if visible_groups is not None:
        keep = [groups.index(g) for g in visible_groups if g in groups]
        if not keep:
            return plt.figure(), [], [], {}
        groups = [groups[i] for i in keep]
        # Filter replicate_data
        new_rep_data = {}
        for (i, j), vals in replicate_data.items():
            if j in keep:
                new_j = keep.index(j)
                new_rep_data[(i, new_j)] = vals
        replicate_data = new_rep_data
    
    # 2) Bands
    want_bands = add_gex_band or add_group_color_band
    bands_frac_top = 0.10 if want_bands else 0.0
    
    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
        color_offset=color_offset,
        palette=palette,
    )
    tp.row_pitch = row_pitch
    if colors is not None:
        tp.colors = [colors.get(tid) for tid in iso_ids]
    
    layout = make_aligned_transcript_panel_layout(
        tp, iso_ids,
        n_panel_cols=len(groups),
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=0.0,
        fig_height=fig_height,
        panel_width=panel_width,
    )
    fig = layout.fig
    
    # Hide colorbar axis
    if layout.ax_cbar is not None:
        layout.ax_cbar.set_visible(False)
        layout.ax_cbar.axis("off")
    
    # 3) Strip version suffixes from isoform labels
    isoform_labels = [_versionless(tid) for tid in iso_ids]
    
    # 4) Draw replicate plot in panel
    ax_panel = layout.ax_panel
    ax_panel.set_axis_off()
    
    n_isoforms = len(iso_ids)
    n_groups_plot = len(groups)
    
    # Create inset for replicate plot - shifted right to avoid overlap with transcripts
    rep_ax_inset = ax_panel.inset_axes([0.0, 0.08, 1.0, 0.92])
    rep_ax_inset.set_axis_off()
    
    # Calculate subplot grid with more spacing between subplots
    subplot_height = 1.0 / n_isoforms

    # Build effective color map for scatter points (independent of add_group_color_band)
    if group_color_map is None:
        _cmap_cat = mpl.cm.get_cmap("tab20")
        _effective_color_map = {
            g: mpl.colors.to_hex(_cmap_cat(idx / max(1, len(groups))))
            for idx, g in enumerate(groups)
        }
    else:
        _effective_color_map = group_color_map

    for i in range(n_isoforms):
        # Create subplot for each isoform with increased spacing
        y_bottom = (n_isoforms - 1 - i) * subplot_height
        
        # Reduced from 0.95 to 0.85 to create more vertical gap between subplots
        gap = subplot_height * 0.10
        sub_ax = rep_ax_inset.inset_axes([0.0, y_bottom + gap, 1.0, subplot_height - gap * 1.5])
        
        # Plot scatter points with jitter
        for j in range(n_groups_plot):
            y = np.asarray(replicate_data.get((i, j), np.array([], float)), float)
            if y.size == 0:
                continue
            x = np.full_like(y, j, dtype=float) + (np.random.random(size=y.size) - 0.5) * jitter_width
            color = _effective_color_map.get(groups[j], "#333333")
            sub_ax.plot(x, np.clip(y, 0.0, 1.0), "o", ms=point_size, alpha=point_alpha, color=color)
        
        # Overlay boxplot if requested
        if box_overlay:
            ys = []
            for j in range(n_groups_plot):
                arr = np.asarray(replicate_data.get((i, j), np.array([], float)), float)
                ys.append(arr if arr.size else np.array([np.nan]))
            
            bp = sub_ax.boxplot(
                ys, positions=np.arange(n_groups_plot), widths=0.55,
                manage_ticks=False, patch_artist=True,
                medianprops=dict(color="#333333", linewidth=1.2),
                boxprops=dict(facecolor="none", edgecolor="#333333", linewidth=1.0),
                whiskerprops=dict(color="#333333", linewidth=1.0),
                capprops=dict(color="#333333", linewidth=1.0),
            )
            for b in bp["boxes"]:
                b.set_alpha(0.55)
        
        # Styling
        sub_ax.set_xlim(-0.5, n_groups_plot - 0.5)
        sub_ax.set_ylim(0.0, 1.0)
        sub_ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4, prune="both"))
        sub_ax.set_ylabel("PSI", fontsize=_fs(13))
        sub_ax.yaxis.set_label_position("right")
        sub_ax.tick_params(axis="y", labelsize=_fs(12), left=False, labelleft=False, right=True, labelright=True)
        sub_ax.tick_params(axis="x", labelsize=_fs(12))
        sub_ax.set_xticks(np.arange(n_groups_plot))
        
        if i == n_isoforms - 1:
            # Only bottom subplot gets x-axis labels
            from allos.quant_plots import _wrap_labels
            if show_xlabels:
                sub_ax.set_xticklabels(
                    _wrap_labels(groups, width=label_wrap),
                    rotation=label_rot,
                    ha='right',
                    rotation_mode='anchor',
                    fontsize=_fs(13)
                )
                if show_xlabel:
                    sub_ax.set_xlabel(group_col, fontsize=_fs(13))
            else:
                sub_ax.set_xticklabels([])
                sub_ax.tick_params(axis='x', which='both', length=0)
        else:
            sub_ax.set_xticklabels([])
        
        # REMOVED: isoform label on right side (redundant with transcript structures on left)
        # The transcript names are already shown with the structure glyphs
    
    # 5) Optional bands
    if want_bands and layout.ax_band_top is not None and len(groups) > 0:
        n_slots = int(add_group_color_band) + int(add_gex_band)
        slot = 0
        
        if add_group_color_band:
            if group_color_map is None:
                cmap_cat = mpl.cm.get_cmap("tab20")
                group_color_map = {
                    g: mpl.colors.to_hex(cmap_cat(i / max(1, len(groups))))
                    for i, g in enumerate(groups)
                }
            add_categorical_band_to_layout(
                layout, groups, color_map=group_color_map,
                slot=slot, n_slots=n_slots, which="top"
            )
            slot += 1
        
        if add_gex_band:
            X_all = _dense_X(adata.X)
            if "geneId" in adata.var:
                m = adata.var["geneId"].astype(str).values == str(gene_id)
            else:
                m = np.zeros(adata.n_vars, bool)
            
            if not np.any(m):
                vals_band = np.zeros(len(groups), float)
            else:
                Xg = X_all[:, m]
                if Xg.ndim == 2 and Xg.shape[1] > 1:
                    Xg = Xg.sum(1)
                lib_tot = X_all.sum(1) + epsilon
                
                all_groups = adata.obs[group_col].astype(str).values
                vals_band = []
                for g in groups:
                    sel = (all_groups == g)
                    if not sel.any():
                        vals_band.append(0.0)
                        continue
                    cp10k = (Xg[sel] / lib_tot[sel]) * 1e4
                    vals_band.append(float(np.nanmean(cp10k)))
                vals_band = np.asarray(vals_band, float)
            
            add_quant_band_to_layout(
                layout, vals_band, cmap="viridis",
                slot=slot, n_slots=n_slots, which="top"
            )
    
    fig.canvas.draw()
    for _ax in fig.axes:
        if _ax.get_axes_locator() is not None:
            _ax.set_position(_ax.get_position())
            _ax.set_axes_locator(None)
    return fig, iso_ids, groups, replicate_data

In [ ]:
#| export
def plot_isoform_violin_composed(
    *,
    transcript_data,
    adata,
    gene_id: str,
    group_col: str,
    gene_col: str = "geneId",
    layer: str | None = None,
    top_n: int | None = None,
    transcripts: List[str] | None = None,
    log1p: bool = True,
    drop_zeros: bool = False,
    min_cells_per_group: int = 10,
    fig_width: float = 18.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    palette: str | list = "tab10",
    stripplot: bool = True,
    point_size: float = 2.0,
    point_alpha: float = 0.4,
    jitter: float = 0.25,
    sharey: bool = False,
    visible_groups: Optional[List[str]] = None,
    add_gex_band: bool = False,
    add_group_color_band: bool = False,
    group_color_map: Optional[Dict[str, str]] = None,
    row_pitch: float = 0.45,
    intron_scale: float = 0.08,  # intron compression (smaller = more compressed)
    color_offset: int = 0,  # rotate transcript color palette by this many steps
) -> Tuple[plt.Figure, List[str], List[str]]:
    """
    Plot violin plots showing isoform expression with transcript structures.
    
    Creates a multi-panel layout with transcript structures on the left and
    violin plots for each isoform-group combination on the right.
    
    Parameters
    ----------
    transcript_data : TranscriptData
        TranscriptData object for gene/transcript lookups
    adata : AnnData
        Annotated data object with transcript counts
    gene_id : str
        Gene ID to plot
    group_col : str
        Column in adata.obs defining groups (e.g., 'cell_type')
    gene_col : str, default "geneId"
        Column in adata.var mapping transcripts to genes
    layer : str, optional
        Layer in adata.layers to use. If None, uses adata.X
    top_n : int, optional
        Number of top isoforms to plot by mean expression.
        Mutually exclusive with `transcripts`.
    transcripts : List[str], optional
        Specific transcript IDs to plot.
        Mutually exclusive with `top_n`.
    log1p : bool, default True
        Apply log1p transformation to counts
    drop_zeros : bool, default False
        Remove zero values before plotting
    min_cells_per_group : int, default 10
        Minimum cells required per group to include
    fig_width : float, default 20.0
        Overall figure width in inches
    draw_cds : bool, default True
        Draw CDS regions on transcript structures
    show_ticks : bool, default True
        Show genomic coordinate ruler
    palette : str or list, default "tab10"
        Color palette for groups (used if group_color_map is not provided)
    stripplot : bool, default True
        Overlay scatter points on violins
    point_size : float, default 2.0
        Size of scatter points
    point_alpha : float, default 0.4
        Alpha transparency of points
    jitter : float, default 0.25
        Horizontal jitter width for points
    sharey : bool, default False
        Share y-axis across all violin panels (default False for better visibility)
    visible_groups : List[str], optional
        Subset of groups to display. If None, all groups are shown.
    add_gex_band : bool, default False
        Add a gene-expression (CP10k) quantitative band above the panel.
    add_group_color_band : bool, default False
        Add colored band showing group categories
    group_color_map : Dict[str, str], optional
        Custom color mapping for groups. If provided, overrides palette for
        both violin colors and group color band.
        
    Returns
    -------
    fig : plt.Figure
        The matplotlib figure
    iso_ids : List[str]
        List of isoform IDs plotted
    groups : List[str]
        List of group names
        
    Examples
    --------
    # Plot top 3 isoforms with violin plots
    fig, iso_ids, groups = plot_isoform_violin_composed(
        transcript_data=td,
        adata=adata,
        gene_id="Myl6",
        group_col="cell_type",
        top_n=3,
        stripplot=True,
    )
    plt.show()
    
    # Plot with custom group colors
    fig, iso_ids, groups = plot_isoform_violin_composed(
        transcript_data=td,
        adata=adata,
        gene_id="Myl6",
        group_col="cell_type",
        top_n=3,
        group_color_map={"type_A": "#FF0000", "type_B": "#00FF00"},
    )
    plt.show()
    """
    import pandas as pd
    import seaborn as sns
    from matplotlib.patches import Rectangle
    
    # Validation
    if (top_n is None) == (transcripts is None):
        raise ValueError("Provide exactly one of: top_n or transcripts")
    
    if group_col not in adata.obs:
        raise ValueError(f"group_col='{group_col}' not found in adata.obs")
    
    if gene_col not in adata.var.columns:
        raise ValueError(f"adata.var['{gene_col}'] not found")
    
    # Determine isoforms
    if transcripts is not None:
        missing = [t for t in transcripts if t not in adata.var_names]
        if missing:
            raise ValueError(f"Transcripts not found in adata.var_names: {missing}")
        iso_ids = transcripts
    else:
        mask = adata.var[gene_col].astype(str).values == str(gene_id)
        iso_ids = adata.var_names[mask].tolist()
        if not iso_ids:
            raise ValueError(f"No isoforms found for gene_id='{gene_id}'")
    
    # Extract expression
    if layer is None:
        Xsub = adata[:, iso_ids].X
    else:
        if layer not in adata.layers:
            raise ValueError(f"layer='{layer}' not found in adata.layers")
        Xsub = adata[:, iso_ids].layers[layer]
    
    Xsub = Xsub.toarray() if hasattr(Xsub, "toarray") else np.asarray(Xsub)
    Xsub = np.asarray(Xsub, dtype=float)
    
    if log1p:
        Xsub = np.log1p(Xsub)
    
    # Top_n selection
    if top_n is not None and len(iso_ids) > int(top_n):
        means = Xsub.mean(axis=0)
        keep = np.argsort(means)[::-1][:int(top_n)]
        iso_ids = [iso_ids[i] for i in keep]
        Xsub = Xsub[:, keep]
    
    # Create long dataframe
    group_vals = adata.obs[group_col].astype(str).values
    df = pd.DataFrame(Xsub, columns=iso_ids)
    df[group_col] = group_vals
    df = df[df[group_col].str.strip() != ""]
    
    long = df.melt(id_vars=group_col, var_name="isoform", value_name="expression")
    
    if drop_zeros:
        long = long[long["expression"] > 0]
    
    # Filter by minimum cells per group
    vc = df[group_col].value_counts()
    keep_groups = vc[vc >= int(min_cells_per_group)].index.tolist()
    long = long[long[group_col].isin(keep_groups)]
    
    # Apply visible_groups filter  -  preserve user-specified order
    if visible_groups is not None:
        keep_groups = [g for g in visible_groups if g in keep_groups]
        long = long[long[group_col].isin(keep_groups)]

    # Remove isoforms with no expression
    long = long.groupby("isoform", observed=False).filter(lambda g: g["expression"].sum() > 0)
    if long.empty:
        raise ValueError("Nothing left to plot after filtering")
    
    groups = list(dict.fromkeys(keep_groups))  # stable order
    long[group_col] = pd.Categorical(long[group_col], categories=groups, ordered=True)
    long["isoform"] = pd.Categorical(long["isoform"], categories=iso_ids, ordered=True)
    
    # Build transcript + panel layout
    want_bands = add_group_color_band or add_gex_band
    n_band_slots = int(add_group_color_band) + int(add_gex_band)
    bands_frac_top = 0.10 * n_band_slots if want_bands else 0.0
    
    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
        color_offset=color_offset,
    )
    tp.row_pitch = row_pitch
    
    layout = make_transcript_panel_layout(
        tp,
        iso_ids,
        n_panel_cols=len(groups),
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=0.0,
    )
    
    # Build color map for violins - use group_color_map if provided, otherwise palette
    if group_color_map is not None:
        color_map = group_color_map
    else:
        if isinstance(palette, str):
            colors = sns.color_palette(palette, len(groups))
        else:
            colors = palette
        color_map = dict(zip(groups, colors))
    
    # Optional bands
    if want_bands and layout.ax_band_top is not None:
        slot = 0
        if add_group_color_band:
            band_color_map = {}
            for g, c in color_map.items():
                band_color_map[g] = c if isinstance(c, str) else mpl.colors.to_hex(c)
            add_categorical_band_to_layout(
                layout, groups, color_map=band_color_map,
                slot=slot, n_slots=n_band_slots, which="top"
            )
            slot += 1
        if add_gex_band:
            vals_band = _compute_gex_vals(adata, gene_id, groups, group_col)
            add_quant_band_to_layout(
                layout, vals_band, cmap="viridis",
                slot=slot, n_slots=n_band_slots, which="top"
            )

    # Access the panel and colorbar axes
    ax_panel = layout.ax_panel
    ax_cbar = layout.ax_cbar
    fig = layout.fig
    
    # Hide the colorbar axis (not needed for violins)
    ax_cbar.set_visible(False)
    ax_panel.set_visible(False)
    
    # Subdivide ax_panel into rows, one for each isoform
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    
    n_isoforms = len(iso_ids)
    isoform_axes = []
    
    # Place violin rows directly in ax_main.transData  -  same coords as transcripts.
    # Rows span exactly row_pitch so they tile flush; separator lines provide visual split.
    ax_panel.set_visible(False)
    violin_x0 = tp.panel_x0
    violin_w  = tp.panel_width
    half_h    = tp.row_pitch / 2  # full height  -  separator at exact midpoint
    pad       = tp.row_pitch * 0.06  # small inner gap from separator line

    for i in range(n_isoforms):
        yc = layout.y_centers[i]
        ax_iso = inset_axes(
            layout.ax_main,
            width="100%",
            height="100%",
            bbox_to_anchor=(violin_x0, yc - half_h + pad, violin_w, half_h * 2 - 2 * pad),
            bbox_transform=layout.ax_main.transData,
            borderpad=0,
        )
        isoform_axes.append(ax_iso)

    # Draw horizontal separator lines between rows (in ax_main data coords)
    for i in range(n_isoforms - 1):
        y_sep = (layout.y_centers[i] + layout.y_centers[i + 1]) / 2
        layout.ax_main.plot(
            [violin_x0, violin_x0 + violin_w], [y_sep, y_sep],
            color='#cccccc', linewidth=0.8, linestyle='-',
            transform=layout.ax_main.transData, zorder=0, clip_on=False,
        )
    
    for i, (iso_id, ax_iso) in enumerate(zip(iso_ids, isoform_axes)):
        iso_data = long[long["isoform"] == iso_id]
        
        if iso_data.empty:
            ax_iso.text(0.5, 0.5, "No data", 
                       ha='center', va='center', transform=ax_iso.transAxes,
                       fontsize=8, color='gray')
            ax_iso.set_xlim(0, 1); ax_iso.set_ylim(0, 1)
            ax_iso.set_xticks([]); ax_iso.set_yticks([])
            continue
        
        sns.violinplot(
            data=iso_data,
            x=group_col,
            y="expression",
            ax=ax_iso,
            order=groups,
            cut=0,
            inner=None,
            density_norm="width",
            palette=color_map,
            linewidth=0.8,
        )
        
        if stripplot:
            sample_data = iso_data.copy()
            if len(sample_data) > 500:
                sample_data = sample_data.sample(n=500, random_state=42)
            sns.stripplot(
                data=sample_data,
                x=group_col,
                y="expression",
                ax=ax_iso,
                order=groups,
                color="k",
                size=float(point_size),
                alpha=float(point_alpha),
                jitter=float(jitter),
            )
        
        ax_iso.set_xlabel("")
        ax_iso.set_ylabel("")
        
        if i < len(iso_ids) - 1:
            ax_iso.set_xticklabels([])
            ax_iso.tick_params(axis='x', which='both', length=0)
        else:
            ax_iso.set_xticklabels(groups, rotation=45, ha='right', rotation_mode='anchor', fontsize=10)
            ax_iso.tick_params(axis='x', which='both', length=0)
        
        ax_iso.tick_params(axis='y', labelsize=10)
        ax_iso.grid(axis='y', alpha=0.3, linewidth=0.5, linestyle='--')
        ax_iso.set_axisbelow(True)
        for spine in ['top', 'right', 'bottom']:
            ax_iso.spines[spine].set_visible(False)
    
    return fig, iso_ids, groups


## Additional Composed Plot Examples
> Demonstration of UMAP and density composed plots for Myl6 and Clta

## Spatial Composed Plot Examples
> Demonstration of spatial composed plots for Myl6 and Clta

In [ ]:
#| export
def plot_isoform_heatmap_percell_composed(
    *,
    transcript_data,
    adata,
    gene_id: str,
    group_col: str,
    top_n: int = 2,
    epsilon: float = 1e-6,
    cell_subset: Optional[List[str]] = None,
    max_cells: Optional[int] = 500,
    cluster_within_groups: bool = True,
    cmap: str = "magma",
    colorbar_label: str = "PSI",
    fig_width: float = 18.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    add_group_color_band: bool = False,
    group_color_map: Optional[Dict[str, str]] = None,
    group_band_position: str = "bottom",
    show_panel_colorbar: bool = False,
    show_group_boundaries: bool = True,
    row_pitch: float = 0.45,
    intron_scale: float = 0.08,  # intron compression (smaller = more compressed)
    color_offset: int = 0,  # rotate transcript color palette by this many steps
) -> Tuple[plt.Figure, List[str], List[str], np.ndarray]:
    """
    Plot a composed per-cell heatmap with transcript structure.

    This function combines transcript structure visualization with a per-cell
    heatmap of isoform PSI values. Cells are sorted by group and optionally
    clustered within each group.

    Parameters
    ----------
    transcript_data : TranscriptData
        Transcript annotation data
    adata : AnnData
        Annotated data object with transcript counts
    gene_id : str
        Gene ID to plot
    group_col : str
        Column in adata.obs for grouping/sorting cells (e.g., 'cell_type')
    top_n : int, default 2
        Number of top isoforms to include
    epsilon : float, default 1e-6
        Small value to avoid division by zero
    cell_subset : List[str], optional
        Specific cell IDs to include
    max_cells : int, default 500
        Maximum number of cells to plot
    cluster_within_groups : bool, default True
        Whether to hierarchically cluster cells within each group
    cmap : str, default "magma"
        Colormap name
    colorbar_label : str, default "PSI"
        Label for colorbar
    fig_width : float, default 20.0
        Figure width in inches
    draw_cds : bool, default True
        Whether to draw CDS regions
    show_ticks : bool, default True
        Whether to show ruler ticks
    add_group_color_band : bool, default False
        Add group color band
    group_color_map : Dict[str, str], optional
        Custom color map for groups
    group_band_position : str, default "bottom"
        Position of group color band ("top" or "bottom")
    show_panel_colorbar : bool, default False
        Whether to show panel colorbar
    show_group_boundaries : bool, default True
        Whether to show vertical lines between groups

    Returns
    -------
    fig : plt.Figure
        The matplotlib figure
    iso_ids : List[str]
        List of isoform IDs
    cell_ids : List[str]
        List of cell IDs (in plot order)
    V : np.ndarray
        PSI matrix (isoforms x cells)

    Examples
    --------
    # Basic per-cell composed heatmap
    fig, iso_ids, cell_ids, V = plot_isoform_heatmap_percell_composed(
        transcript_data=td,
        adata=adata,
        gene_id="Myl6",
        group_col="cell_type",
        top_n=3,
        max_cells=200,
    )
    plt.show()

    # With group color bands
    fig, iso_ids, cell_ids, V = plot_isoform_heatmap_percell_composed(
        transcript_data=td,
        adata=adata,
        gene_id="Clta",
        group_col="cell_type",
        top_n=2,
        add_group_color_band=True,
        max_cells=300,
    )
    plt.show()
    """
    from scipy.cluster.hierarchy import linkage, leaves_list
    from scipy.spatial.distance import pdist
    from matplotlib.patches import Rectangle

    if "geneId" not in adata.var:
        raise ValueError("adata.var must contain 'geneId' column")

    if group_col not in adata.obs:
        raise ValueError(f"group_col '{group_col}' not found in adata.obs")

    # Get isoforms for this gene
    mask = adata.var["geneId"].astype(str).values == str(gene_id)
    if not np.any(mask):
        return plt.figure(), [], [], np.zeros((0, 0), float)

    # Subset to specific cells if requested
    if cell_subset is not None:
        adata = adata[cell_subset, :].copy()

    # Sample cells within each group if too many
    if max_cells is not None and adata.n_obs > max_cells:
        groups = adata.obs[group_col].astype(str).values
        unique_groups = list(dict.fromkeys(groups))
        cells_per_group = max(1, max_cells // len(unique_groups))

        np.random.seed(42)
        keep_idx = []
        for g in unique_groups:
            g_idx = np.where(groups == g)[0]
            if len(g_idx) > cells_per_group:
                g_idx = np.random.choice(g_idx, size=cells_per_group, replace=False)
            keep_idx.extend(g_idx)
        adata = adata[keep_idx, :].copy()

    # Extract expression matrix
    X = _dense_X(adata.X)
    iso_ids_all = adata.var_names[mask]
    G = X[:, mask]  # cells x isoforms-of-gene

    # Calculate per-cell PSI
    lib = G.sum(1) + float(epsilon)
    PSI = np.divide(
        G,
        lib[:, None],
        out=np.zeros_like(G, float),
        where=(lib[:, None] > 0),
    )

    # Select top isoforms by mean PSI
    if top_n is not None and top_n < len(iso_ids_all):
        mean_psi = PSI.mean(0)
        keep_idx = np.argsort(mean_psi)[::-1][:top_n]
        iso_ids = iso_ids_all[keep_idx].tolist()
        PSI = PSI[:, keep_idx]
    else:
        iso_ids = iso_ids_all.tolist()

    # Sort cells by group, with optional clustering within groups
    groups = adata.obs[group_col].astype(str).values
    unique_groups = list(dict.fromkeys(groups))

    cell_order = []
    group_boundaries = [0]

    for g in unique_groups:
        g_idx = np.where(groups == g)[0]

        if cluster_within_groups and len(g_idx) > 2:
            # Hierarchical clustering within group
            psi_subset = PSI[g_idx, :]

            # Only cluster if we have variance
            if psi_subset.std() > 1e-6:
                try:
                    dist = pdist(psi_subset, metric='euclidean')
                    if len(dist) > 0 and not np.all(dist == 0):
                        Z = linkage(dist, method='average')
                        cluster_order = leaves_list(Z)
                        g_idx = g_idx[cluster_order]
                except:
                    pass  # Keep original order if clustering fails

        cell_order.extend(g_idx)
        group_boundaries.append(len(cell_order))

    cell_order = np.array(cell_order)

    # Reorder PSI matrix
    V = PSI[cell_order, :].T  # isoforms x cells
    cell_ids = adata.obs_names[cell_order].tolist()
    cell_groups = groups[cell_order]
    n_cells = len(cell_ids)

    # Decide band placement
    bands_frac_top = 0.0
    bands_frac_bottom = 0.0

    # Build transcript + panel layout
    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
        palette=palette,
        color_offset=color_offset,
    )
    tp.row_pitch = row_pitch

    layout = make_transcript_panel_layout(
        tp,
        iso_ids,
        n_panel_cols=n_cells,
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=bands_frac_bottom,
    )

    # Main heatmap panel - with empty column labels since too many cells
    heat_meta = add_heatmap_to_layout(
        tp,
        layout,
        V,
        cmap=cmap,
        colorbar_label=colorbar_label,
        col_labels=[""] * n_cells,  # Don't show individual cell labels
        label_wrap=14,
        label_rot=45,
        show_panel_colorbar=show_panel_colorbar,
    )
    psi_norm = mpl.colors.Normalize(vmin=heat_meta["vmin"], vmax=heat_meta["vmax"])
    psi_cmap = mpl.cm.get_cmap(heat_meta["cmap"])

    # Get the panel axis for adding custom elements
    fig = layout.fig
    ax = layout.ax_panel
    n_rows = len(iso_ids)
    n_cols = len(cell_ids)

    # Set axis limits explicitly (don't call tight_layout on composed plots!)
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(-0.5, n_rows - 0.5)

    # Create color palette for groups (for legend and boundaries)
    cat_used = False
    if group_color_map is None:
        cmap_cat = mpl.cm.get_cmap("tab10")
        group_color_map = {
            g: mpl.colors.to_hex(cmap_cat(idx % 10))
            for idx, g in enumerate(unique_groups)
        }
    cat_used = True

    # Add group boundaries and colored bar (AFTER tight_layout)
    if show_group_boundaries and len(group_boundaries) > 2:
        # Draw white boundary lines
        for boundary in group_boundaries[1:-1]:
            ax.axvline(boundary - 0.5, color="white", linewidth=2.5, alpha=1.0, zorder=10)

        # Draw thin colored bar directly above heatmap
        box_height = 0.06

        for i in range(len(group_boundaries) - 1):
            start = group_boundaries[i]
            end = group_boundaries[i + 1]
            group_name = cell_groups[start]

            # Draw thin colored rectangle directly above heatmap
            color = group_color_map.get(group_name, "gray")
            rect = Rectangle(
                (start - 0.5, n_rows - 0.5),
                end - start,
                box_height,
                facecolor=color, edgecolor="none", alpha=0.7,
                clip_on=False,
                transform=ax.transData,
                zorder=15,
            )
            ax.add_patch(rect)

        # Add labels below heatmap with counts
        for i in range(len(group_boundaries) - 1):
            start = group_boundaries[i]
            end = group_boundaries[i + 1]
            mid = (start + end) / 2
            group_name = cell_groups[start]
            n_cells_in_group = end - start

            group_label = f"{group_name}\n(n={n_cells_in_group})"
            ax.text(
                mid, -0.7, group_label,
                ha="center", va="top", fontsize=8,
                rotation=45, fontstyle="italic",
                transform=ax.transData
            )

    # Add title
    suffix = " - clustered" if cluster_within_groups else ""
    title_str = f"{gene_id} - Per-Cell (n={n_cols}{suffix})"
    fig.suptitle(title_str, fontsize=14, fontweight="bold", y=0.98)

    # ---- legend panel (matching heatmap composed structure) ----
    axL = getattr(layout, "ax_legend", None)
    if axL is not None and (cat_used or psi_norm is not None):
        axL.clear()
        axL.set_axis_off()
        from matplotlib.lines import Line2D
        from mpl_toolkits.axes_grid1.inset_locator import inset_axes

        # PSI colorbar only (no groups legend - color band at top is sufficient)
        if psi_norm is not None and psi_cmap is not None:
            psi_ax = inset_axes(
                axL,
                width="40%",
                height="50%",
                loc="center",
                bbox_to_anchor=(0.1, 0.2, 0.8, 0.6),
                bbox_transform=axL.transAxes,
                borderpad=0,
            )
            cb_psi = mpl.colorbar.ColorbarBase(
                psi_ax,
                cmap=psi_cmap,
                norm=psi_norm,
                orientation="vertical",
            )
            cb_psi.set_label("PSI", fontsize=8)
            cb_psi.ax.tick_params(labelsize=7)

    return fig, iso_ids, cell_ids, V


In [ ]:
#| export
def plot_isoform_stacked_bar_composed(
    *,
    transcript_data,
    adata,
    gene_id: str,
    group_col: str,
    top_n: Optional[int] = None,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    compute_on_selection_only: bool = False,
    visible_groups: Optional[List[str]] = None,
    fig_width: float = 18.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    label_wrap: int = 14,
    label_rot: int = 45,
    show_values: bool = False,
    show_title: bool = False,
    add_group_color_band: bool = False,
    group_color_map: Optional[Dict[str, str]] = None,
    add_gex_band: bool = False,
    add_gex_bar: bool = False,
    gex_bar_height: float = 0.003,
    gex_legend_width: float = 0.35,
    gex_legend_x: float = 0.15,
    row_pitch: float = 0.45,
    intron_scale: float = 0.08,  # intron compression  -  matches standalone default
    color_offset: int = 0,  # rotate transcript color palette by this many steps
    palette: str = "wong",       # 'wong' | 'tol' | 'ghibli' | 'tab10'
    font_scale: float = 1.0,  # scale all font sizes uniformly
    colors: Optional[Dict[str, str]] = None,  # isoform_id -> hex color overrides
    fig_height: float | None = None,  # force canvas height for alignment across plot types
    show_xlabels: bool = True,  # set False on top panels when stacking in Illustrator
    panel_width: float | None = None,  # fix data-panel width for cross-plot alignmentor
) -> Tuple[plt.Figure, List[str], List[str], np.ndarray]:
    """
    Plot composed figure with transcript structures and stacked bar chart.

    Creates a two-panel figure showing:
    - Left: Transcript structures for isoforms
    - Right: Stacked bar chart showing isoform composition per group

    Colors are matched between transcript structures and bar segments.

    Parameters
    ----------
    transcript_data : TranscriptData
        Transcript annotation data
    adata : AnnData
        Annotated data object with transcript counts
    gene_id : str
        Gene ID to plot
    group_col : str
        Column in adata.obs defining groups (e.g., 'cell_type')
    top_n : int, optional
        Number of top isoforms to show separately. Others grouped as "Other".
        If None, shows all isoforms.
    estimator : str, default "pseudobulk"
        PSI estimator ('pseudobulk', 'dirichlet', 'cell-mean', 'cell-median', 'coverage-weighted')
    dirichlet_alpha : float, default 0.5
        Dirichlet alpha parameter
    epsilon : float, default 1e-6
        Small value to avoid division by zero
    compute_on_selection_only : bool, default False
        If True, compute PSI only on selected top_n isoforms (rescaled to sum to 100%).
        If False, compute PSI on all isoforms and show remaining as "Other" category.
    visible_groups : List[str], optional
        Subset of groups to display. If None, all groups are shown.
    fig_width : float, default 14.0
        Figure width in inches
    draw_cds : bool, default True
        Whether to draw CDS regions in transcript structures
    show_ticks : bool, default True
        Whether to show ruler ticks
    label_wrap : int, default 14
        Width for wrapping x-axis labels
    label_rot : int, default 45
        Rotation angle for x-axis labels
    show_values : bool, default False
        Whether to show percentage values on bars
    show_title : bool, default False
        Whether to show the figure title
    add_group_color_band : bool, default False
        Add a categorical color band above the panel showing group identity.
    group_color_map : Dict[str, str], optional
        Custom color mapping for groups used in the color band.
    add_gex_band : bool, default False
        Add a gene-expression (CP10k) quantitative band above the panel.
    add_gex_bar : bool, default False
        Whether to add gene expression bar above stacked bars
    gex_bar_height : float, default 0.003
        Fixed height of gene expression bar as fraction of main plot height
    gex_legend_width : float, default 0.35
        Width of the GEX legend colorbar (as fraction, 0-1)
    gex_legend_x : float, default 0.15
        Horizontal position of GEX legend colorbar (0=left, 1=right)

    Returns
    -------
    fig : plt.Figure
        The matplotlib figure
    iso_ids : List[str]
        List of isoform IDs (including "Other" if top_n used)
    groups : List[str]
        List of group names
    V : np.ndarray
        PSI matrix (isoforms x groups)

    Examples
    --------
    # Basic composed stacked bar chart with "Other" category
    fig, iso_ids, groups, V = plot_isoform_stacked_bar_composed(
        transcript_data=td,
        adata=adata,
        gene_id="Myl6",
        group_col="cell_type",
        top_n=5,
        compute_on_selection_only=False,
    )
    plt.show()
    font_scale : float, default 1.0  -  multiply all font sizes by this factor.
    colors : Dict[str, str], optional  -  isoform_id -> hex color overrides.
    """
    from allos.quant_plots import _wrap_labels
    _fs = lambda x: int(round(x * font_scale))

    # Compute PSI matrix (get all isoforms first)
    result = _select_isoforms(
        adata, gene_id, group_col,
        top_n=None, estimator=estimator,
        dirichlet_alpha=dirichlet_alpha, epsilon=epsilon,
        visible_groups=visible_groups,
    )
    if result is None:
        return plt.figure(), [], [], np.zeros((0, 0), float)
    iso_ids, groups, V = result

    # Compute gene expression if requested (add_gex_bar or add_gex_band)
    gex_vals = None
    gex_norm = None
    gex_cmap = None
    if add_gex_bar or add_gex_band:
        gex_vals = _compute_gex_vals(adata, gene_id, groups, group_col, epsilon=epsilon)
        gex_norm = mpl.colors.Normalize(vmin=float(gex_vals.min()), vmax=float(gex_vals.max()))
        gex_cmap = mpl.cm.get_cmap("viridis")

    # Handle top_n filtering with or without "Other" category
    has_other = False
    if top_n is not None and len(iso_ids) > int(top_n):
        mean_psi = V.mean(axis=1)
        top_idx = np.argsort(mean_psi)[::-1][:int(top_n)]
        top_iso_ids = [iso_ids[i] for i in top_idx]
        V_top = V[top_idx, :]

        if compute_on_selection_only:
            iso_ids_for_bars = top_iso_ids
            iso_ids_for_transcripts = top_iso_ids
            row_sums = V_top.sum(axis=0, keepdims=True)
            V_for_bars = np.divide(
                V_top, row_sums,
                out=np.zeros_like(V_top, float),
                where=(row_sums > 0),
            )
            has_other = False
        else:
            other_idx = [i for i in range(len(iso_ids)) if i not in top_idx]
            if other_idx:
                V_other = V[other_idx, :].sum(axis=0, keepdims=True)
                iso_ids_for_bars = top_iso_ids + ["Other"]
                V_for_bars = np.vstack([V_top, V_other])
                has_other = True
            else:
                iso_ids_for_bars = top_iso_ids
                V_for_bars = V_top
            iso_ids_for_transcripts = top_iso_ids
    else:
        iso_ids_for_bars = iso_ids
        iso_ids_for_transcripts = iso_ids
        V_for_bars = V

    # Build transcript panel layout
    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
        palette=palette,
        color_offset=color_offset,
    )
    tp.row_pitch = row_pitch

    # Assign isoform colors  -  palette resolved from name
    _palette_map = {"wong": _PALETTE_WONG, "tol": _PALETTE_TOL_MUTED, "ghibli": list(_PALETTE_GHIBLI), "tab10": _DEFAULT_COLORS}
    _base_pal = _palette_map.get(palette, _PALETTE_WONG) if isinstance(palette, str) else list(palette)
    _n = color_offset % len(_base_pal)
    _palette = _base_pal[_n:] + _base_pal[:_n]
    isoform_colors = {}
    tp.colors = []
    for i, iso_id in enumerate(iso_ids_for_transcripts):
        color = colors.get(iso_id, _palette[i % len(_palette)]) if colors else _palette[i % len(_palette)]
        isoform_colors[iso_id] = color
        tp.colors.append(color)
    if has_other:
        isoform_colors["Other"] = "#999999"

    # Bands
    want_bands = add_group_color_band or add_gex_band
    n_band_slots = int(add_group_color_band) + int(add_gex_band)
    bands_frac_top = 0.10 * n_band_slots if want_bands else 0.0

    layout = make_aligned_transcript_panel_layout(
        tp,
        iso_ids_for_transcripts,
        n_panel_cols=len(groups),
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=bands_frac_top,
        bands_frac_bottom=0.0,
        fig_height=fig_height,
        panel_width=panel_width,
    )

    if hasattr(layout, 'ax_cbar') and layout.ax_cbar is not None:
        layout.ax_cbar.set_visible(False)

    ax_bar = layout.ax_panel
    fig = layout.fig

    # Optional bands
    if want_bands and layout.ax_band_top is not None:
        slot = 0
        if add_group_color_band:
            band_cm = group_color_map if group_color_map else {}
            add_categorical_band_to_layout(
                layout, groups, color_map=band_cm,
                slot=slot, n_slots=n_band_slots, which="top"
            )
            slot += 1
        if add_gex_band and gex_vals is not None:
            add_quant_band_to_layout(
                layout, gex_vals, cmap="viridis",
                slot=slot, n_slots=n_band_slots, which="top"
            )

    # Draw stacked bar chart
    x = np.arange(len(groups))
    bottoms = np.zeros(len(groups))

    for i, iso_id in enumerate(iso_ids_for_bars):
        heights = V_for_bars[i, :] * 100
        color = isoform_colors[iso_id]
        ax_bar.bar(x, heights, width=0.95, bottom=bottoms,
                   color=color, edgecolor='white', linewidth=0.5, align='center')
        if show_values:
            for j, (xpos, height, bottom) in enumerate(zip(x, heights, bottoms)):
                if height > 2:
                    ax_bar.text(xpos, bottom + height / 2, f'{height:.1f}%',
                                ha='center', va='center', fontsize=_fs(11),
                                color='white', fontweight='bold')
        bottoms += heights

    ax_bar.set_xlim(-0.5, len(groups) - 0.5)
    ax_bar.set_ylim(0, 100)

    # Add gene expression bar above stacked bars if requested
    if add_gex_bar and gex_vals is not None:
        gex_bar_fixed_height = gex_bar_height * 100
        for j, (xpos, gex_val) in enumerate(zip(x, gex_vals)):
            color = gex_cmap(gex_norm(gex_val))
            ax_bar.bar(xpos, gex_bar_fixed_height, width=0.95, bottom=100,
                       color=color, edgecolor='none', linewidth=0,
                       alpha=1.0, clip_on=False, zorder=10)
        ax_bar.axhline(100, color='black', linewidth=1.0, linestyle='-', zorder=5)

    ax_bar.set_xticks(x)
    if show_xlabels:
        ax_bar.set_xticklabels(
            _wrap_labels(groups, width=label_wrap),
            rotation=label_rot, ha='right', rotation_mode='anchor', fontsize=_fs(11)
        )
    else:
        ax_bar.set_xticklabels([])
        ax_bar.tick_params(axis='x', which='both', length=0)
    ax_bar.tick_params(axis='y', left=False, labelleft=False)
    ax_bar.yaxis.grid(True, alpha=0.3, linestyle='--', linewidth=0.5, zorder=0)
    ax_bar.set_axisbelow(True)
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.spines['left'].set_visible(False)
    ax_bar.spines['bottom'].set_visible(True)

    # Legend panel  -  blank by default; shows GEX colorbar when add_gex_bar=True
    axL = getattr(layout, 'ax_legend', None)
    if axL is not None:
        axL.set_visible(True)  # always visible to maintain figure width
        axL.clear()
        axL.set_axis_off()
        if add_gex_bar and gex_norm is not None and gex_cmap is not None:
            from mpl_toolkits.axes_grid1.inset_locator import inset_axes
            gex_ax = inset_axes(
                axL, width=f"{gex_legend_width*100:.0f}%", height='70%',
                loc='center left',
                bbox_to_anchor=(gex_legend_x, 0.15, gex_legend_width, 0.70),
                bbox_transform=axL.transAxes, borderpad=0,
            )
            cb_gex = mpl.colorbar.ColorbarBase(
                gex_ax, cmap=gex_cmap, norm=gex_norm, orientation='vertical'
            )
            cb_gex.set_label('GEX\n(CP10k)', fontsize=_fs(12))
            cb_gex.ax.tick_params(labelsize=_fs(11))

    if show_title:
        fig.suptitle(f'{gene_id} - Isoform Composition', fontsize=14, fontweight='bold')

    fig.canvas.draw()
    for _ax in fig.axes:
        if _ax.get_axes_locator() is not None:
            _ax.set_position(_ax.get_position())
            _ax.set_axes_locator(None)
    return fig, iso_ids_for_bars, groups, V_for_bars


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| export
def plot_isoform_spatial_composed_HD(
    *,
    transcript_data,
    adata,
    gene_id: str,
    top_n: int = 2,
    fig_width: float = 28.0,
    draw_cds: bool = True,
    show_ticks: bool = True,
    show_individual_colorbars: bool = True,
    spatial_cmap: Optional[str] = None,
    spatial_cmaps: Optional[List[str]] = None,
    spatial_max_cols: int = 2,
    spatial_size: float = 5.0,
    spatial_alpha: float = 0.9,
    invert_y: bool = True,
    aspect_equal: bool = True,
    color_offset: int = 0,
    row_pitch: float = 0.45,
    intron_scale: float = 0.08,  # intron compression (smaller = more compressed)
    layer: Optional[str] = None,
    transcripts: Optional[List[str]] = None,
    log_scale: bool = True,
    norm_per_spot: bool = False,
    norm_target: float = 1e4,
    hd_marker: str = "s",
    hd_bg_color: str = "#E6E6E6",
    hd_bg_alpha: float = 0.35,
    hd_bg_size_factor: float = 1.0,
    bin_factor: Optional[int] = None,
    rotate: float = 0,
) -> Tuple[plt.Figure, List[str]]:
    """
    Composed plot for Visium HD: transcript structure glyphs + spatial scatter panels.

    Renders square-bin scatter on a white background (visium_like style), using
    array_col/array_row from adata.obs for spatial coordinates.

    Parameters
    ----------
    top_n : int
        Number of top-expressed isoforms to show (by total expression).
    layer : str, optional
        Layer to read expression from. Defaults to 'counts' if present.
    transcripts : list of str, optional
        Explicit transcript IDs  -  bypasses top_n selection.
    log_scale : bool
        Use LogNorm colorscale. Default True for visium_like style.
    norm_per_spot : bool
        Normalise by per-spot library size before plotting.
    norm_target : float
        CPM target when norm_per_spot=True (default 1e4).
    spatial_cmap : str, optional
        Single colormap applied to all panels (e.g. "Reds", "magma").
        spatial_cmaps takes precedence if also provided.
    spatial_cmaps : list of str, optional
        Per-isoform colormap list; overrides spatial_cmap.
    aspect_equal : bool
        If True (default), each spatial subplot uses equal x/y scaling,
        preserving true tissue geometry.
    color_offset : int
        Rotate transcript color palette by this many steps.
    """
    import pandas as _pd
    from matplotlib.colors import LogNorm

    # 1) Auto-detect layer
    if layer is None and "counts" in adata.layers:
        layer = "counts"

    # 2) Resolve isoforms
    if transcripts is not None:
        iso_ids = list(transcripts)
    else:
        if "geneId" not in adata.var.columns:
            raise ValueError("adata.var must contain a 'geneId' column.")
        mask = adata.var["geneId"].astype(str).values == str(gene_id)
        iso_all = adata.var_names[mask]
        if len(iso_all) == 0:
            return plt.figure(), []
        _Xg = adata[:, iso_all].layers[layer] if layer else adata[:, iso_all].X
        totals = np.asarray(_Xg.sum(0)).ravel() if issparse(_Xg) else np.asarray(_Xg, float).sum(0)
        iso_ids = iso_all[np.argsort(totals)[::-1][:top_n]].tolist()

    if not iso_ids:
        return plt.figure(), []

    n_iso = len(iso_ids)

    # 3) Build layout
    tp = TranscriptPlots(
        transcript_data=transcript_data,
        intron_scale=intron_scale,
        intron_scale_mode="relative",
        exon_scale_mode="none",
        show_length=False,
        color_offset=color_offset,
    )
    tp.row_pitch = row_pitch

    layout = make_transcript_panel_layout(
        tp, iso_ids,
        n_panel_cols=max(10, n_iso),
        fig_width=fig_width,
        draw_cds=draw_cds,
        show_ruler=show_ticks,
        bands_frac_top=0.0,
        bands_frac_bottom=0.0,
    )
    fig = layout.fig
    if layout.ax_cbar is not None:
        layout.ax_cbar.set_visible(False)
        layout.ax_cbar.axis("off")

    # 4) Spatial coordinates from HD array lattice
    if "array_row" not in adata.obs.columns or "array_col" not in adata.obs.columns:
        raise ValueError("adata.obs must contain 'array_row' and 'array_col' columns.")
    coords = np.c_[
        adata.obs["array_col"].to_numpy(dtype=float),
        adata.obs["array_row"].to_numpy(dtype=float),
    ]

    # 5) Extract expression (sparse-aware)
    Xs = adata[:, iso_ids].layers[layer] if layer else adata[:, iso_ids].X
    expr = np.empty((n_iso, adata.n_obs), dtype=float)
    for j in range(n_iso):
        col_j = Xs[:, j]
        expr[j] = (
            np.asarray(col_j.todense()).ravel() if issparse(col_j)
            else np.asarray(col_j, float).ravel()
        )
    if norm_per_spot:
        _src = adata.layers[layer] if layer else adata.X
        _lib = np.asarray(_src.sum(1), float).ravel() + 1e-6
        expr = (expr / _lib[np.newaxis, :]) * norm_target

    # Resolve colormap list: spatial_cmaps > spatial_cmap > default
    if spatial_cmaps is None:
        spatial_cmaps = [spatial_cmap or "viridis"] * n_iso
    if len(spatial_cmaps) != n_iso:
        raise ValueError("spatial_cmaps must be None or same length as iso_ids.")

    # 6) Tile spatial panels
    ax_panel = layout.ax_panel
    ax_panel.set_axis_off()
    ncols = min(spatial_max_cols, n_iso)
    nrows = int(np.ceil(n_iso / ncols))
    cell_w, cell_h = 1.0 / ncols, 1.0 / nrows
    pad_x, pad_y = 0.15, 0.10

    for i, (tr, cmap_name, vals) in enumerate(zip(iso_ids, spatial_cmaps, expr)):
        row, col = i // ncols, i % ncols
        n_in_row = min(ncols, n_iso - row * ncols)
        x_offset = (ncols - n_in_row) * cell_w / 2.0 if n_in_row < ncols else 0.0

        ax = ax_panel.inset_axes([
            x_offset + col * cell_w + pad_x * cell_w,
            (nrows - 1 - row) * cell_h + pad_y * cell_h,
            cell_w * (1 - 2 * pad_x),
            cell_h * (1 - 2 * pad_y),
        ])
        ax.set_facecolor("white")

        if bin_factor is not None:
            bx = np.floor(coords[:, 0] / bin_factor).astype(int)
            by = np.floor(coords[:, 1] / bin_factor).astype(int)
            _df = _pd.DataFrame({"bx": bx, "by": by, "v": vals})
            _agg = _df.groupby(["bx", "by"])["v"].sum()
            _cx = (_agg.index.get_level_values("bx") + 0.5) * bin_factor
            _cy = (_agg.index.get_level_values("by") + 0.5) * bin_factor
            _plot_coords = np.c_[np.asarray(_cx), np.asarray(_cy)]
            _plot_vals   = np.asarray(_agg)
            _s = spatial_size * max(1.0, bin_factor ** 2 * 0.5)
        else:
            _plot_coords = coords
            _plot_vals   = vals
            _s = spatial_size * hd_bg_size_factor

        if rotate:
            _theta = np.radians(rotate)
            _R = np.array([[np.cos(_theta), -np.sin(_theta)], [np.sin(_theta), np.cos(_theta)]])
            _plot_coords = _plot_coords @ _R.T

        _nz = _plot_vals[_plot_vals > 0]
        panel_vmin, panel_vmax = 0.0, float(np.nanpercentile(_nz, 99.0)) if len(_nz) else 1.0

        norm_obj = None
        if log_scale and len(_nz):
            _lo = 1e-3
            norm_obj = LogNorm(vmin=_lo, vmax=max(panel_vmax, _lo * 10))

        mask_expr = _plot_vals > 0

        # Faint background tissue bins
        if (~mask_expr).any():
            ax.scatter(
                _plot_coords[~mask_expr, 0], _plot_coords[~mask_expr, 1],
                c=hd_bg_color, s=_s,
                marker=hd_marker, alpha=hd_bg_alpha, linewidths=0, rasterized=True,
            )

        # Expressing bins
        mappable = ax.scatter(
            _plot_coords[mask_expr, 0] if mask_expr.any() else [],
            _plot_coords[mask_expr, 1] if mask_expr.any() else [],
            c=_plot_vals[mask_expr] if mask_expr.any() else [],
            s=_s, marker=hd_marker, cmap=cmap_name,
            norm=norm_obj,
            vmin=None if norm_obj else panel_vmin,
            vmax=None if norm_obj else panel_vmax,
            alpha=spatial_alpha, linewidths=0, rasterized=True,
        )

        if invert_y:
            ax.invert_yaxis()

        # Preserve true tissue geometry
        if aspect_equal:
            ax.set_aspect('equal', adjustable='box')

        show_xlab, show_ylab = row == nrows - 1, col == 0
        ax.set_xlabel("X" if show_xlab else "", fontsize=8)
        ax.set_ylabel("Y" if show_ylab else "", fontsize=8)
        ax.tick_params(axis="x", labelbottom=show_xlab, bottom=True, length=2, width=0.5)
        ax.tick_params(axis="y", labelleft=show_ylab, left=True, length=2, width=0.5)
        ax.set_title(tr.split(".")[0], fontsize=9)

        if show_individual_colorbars and mask_expr.any():
            cax = ax.inset_axes([1.02, 0.1, 0.04, 0.8])
            fig.colorbar(mappable, cax=cax, orientation="vertical").ax.tick_params(labelsize=6)

    return fig, iso_ids

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()